# 03. Временные, сетевые и событийные признаки ЭЭГ при ЧМТ

## Назначение ноутбука

Данный ноутбук рассчитывает признаки ЭЭГ, которые дополняют спектральный анализ:

1. DFA / LRTC — долговременная временная структура амплитудных огибающих;
2. burst-анализ — транзиентная организация ритмической активности;
3. функциональная связность между ROI;
4. сетевые метрики;
5. событийные признаки;
6. объединённая таблица признаков на уровне записи.

Входом является таблица `analysis_ready_inventory_cleaned.csv`, сформированная в ноутбуке предобработки.

Основной выходной файл:

```text
temporal_connectivity_event_features_record_level.csv
```

Эта таблица затем объединяется со спектральными признаками из `02_spectral_features.ipynb`.

## Связь с предыдущими ноутбуками

```text
01_prepare_eeg_dataset.ipynb
        ↓
analysis_ready_inventory_cleaned.csv
        ↓
02_spectral_features.ipynb
        ↓
spectral_features_record_level.csv

01_prepare_eeg_dataset.ipynb
        ↓
analysis_ready_inventory_cleaned.csv
        ↓
03_temporal_connectivity_events.ipynb
        ↓
temporal_connectivity_event_features_record_level.csv
```

Текущий ноутбук не повторяет расчёт PSD и bandpower. Его задача — получить временные, сетевые и событийные признаки, которые затем будут объединены со спектральными признаками.

In [ ]:
# @title 0.1. Установка зависимостей

!pip -q install mne scipy pandas numpy matplotlib tqdm scikit-learn statsmodels networkx

In [ ]:
# @title 0.2. Импорты, стиль графиков и базовые настройки

import gc
import json
import logging
import math
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.signal as spsig
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from scipy.spatial.distance import squareform
from tqdm import tqdm

import mne

warnings.filterwarnings("ignore")

# Единые цвета групп во всех ноутбуках проекта.
COL_TBI = "#9B1B30"
COL_CONTROL = "#1F3F7A"

GROUP_COLORS = {
    "TBI": COL_TBI,
    "Control": COL_CONTROL,
}

GROUP_LABELS_RU = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
}

plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)


def save_figure(fig, output_path: Path) -> None:
    """
    Сохраняет рисунок с аккуратными отступами.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Объект рисунка.
    output_path : Path
        Путь для сохранения изображения.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    print(f"Рисунок сохранён: {output_path}")


def save_table(df: pd.DataFrame, output_path: Path) -> None:
    """
    Сохраняет таблицу в CSV-файл.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица для сохранения.
    output_path : Path
        Путь для сохранения CSV-файла.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Таблица сохранена: {output_path}")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


# 1. Конфигурация анализа

В этом разделе задаются пути к подготовленным данным, параметры временного анализа, burst-детекции, функциональной связности и событийного анализа.

Рабочая папка по умолчанию соответствует второму варианту предобработки с ICA/ICLabel/AutoReject:

```text
/content/drive/MyDrive/VKR/TBI/work-ica-ar
```

In [ ]:
# @title 1.1. Конфигурация временного, сетевого и событийного анализа

CONFIG = {
    # ------------------------------------------------------------------
    # Пути
    # ------------------------------------------------------------------
    "work_dir": Path("/content/drive/MyDrive/VKR/TBI/work-3"),

    "analysis_ready_inventory": (
        Path("/content/drive/MyDrive/VKR/TBI/work-3")
        / "output"
        / "tables"
        / "analysis_ready_inventory_cleaned.csv"
    ),

    # ------------------------------------------------------------------
    # Подвыборка эпох для сопоставимости длительности записей
    # ------------------------------------------------------------------
    "max_epochs_per_record": 60,
    "epoch_random_state": 97,

    # ------------------------------------------------------------------
    # Частотные диапазоны для временных признаков
    # ------------------------------------------------------------------
    "temporal_bands": {
        "alpha": (8.0, 13.0),
        "beta": (13.0, 30.0),
    },

    # ------------------------------------------------------------------
    # ROI
    # ------------------------------------------------------------------
    "roi_channels": {
        "frontal": [
            "Fp1", "Fp2", "AF3", "AF4", "AF7", "AF8",
            "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "Fz",
        ],
        "central": [
            "C1", "C2", "C3", "C4", "C5", "C6", "Cz",
            "FC1", "FC2", "FC3", "FC4", "FC5", "FC6",
            "CP1", "CP2", "CP3", "CP4", "CP5", "CP6", "CPz",
        ],
        "temporal": [
            "T7", "T8", "FT7", "FT8", "TP7", "TP8",
        ],
        "parietal": [
            "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "Pz",
            "PO3", "PO4", "PO7", "PO8", "POz",
        ],
        "occipital": [
            "O1", "O2", "Oz",
        ],
    },

    # ------------------------------------------------------------------
    # DFA
    # ------------------------------------------------------------------
    "dfa_min_window_sec": 0.5,
    "dfa_max_window_sec": 8.0,
    "dfa_n_windows": 12,

    # ------------------------------------------------------------------
    # Burst
    # ------------------------------------------------------------------
    "burst_z_threshold": 2.0,
    "burst_min_duration_sec": 0.08,
    "burst_merge_gap_sec": 0.04,

    # ------------------------------------------------------------------
    # Connectivity
    # ------------------------------------------------------------------
    "connectivity_bands": {
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta": (13.0, 30.0),
    },
    "connectivity_methods": [
        "coh",
        "plv",
        "imcoh",
        "wpli",
    ],

    # ------------------------------------------------------------------
    # Event detection
    # ------------------------------------------------------------------
    "event_band": (1.0, 30.0),
    "event_z_threshold": 3.0,
    "event_min_duration_sec": 0.08,
    "event_merge_gap_sec": 0.04,

    # ------------------------------------------------------------------
    # Статистика
    # ------------------------------------------------------------------
    "alpha": 0.05,
    "random_state": 97,

    # ------------------------------------------------------------------
    # Кэширование расчётных таблиц
    # ------------------------------------------------------------------
    # Если True и таблица уже существует на диске, ячейка считывает её
    # вместо повторного тяжёлого расчёта. Для полного пересчёта поставьте False.
    "use_cached_tables": True,
}

OUTPUT_DIR = CONFIG["work_dir"] / "output_temporal_connectivity_events"

OUT = {
    "tables": OUTPUT_DIR / "tables",
    "figures": OUTPUT_DIR / "figures",
    "logs": OUTPUT_DIR / "logs",
    "cache": OUTPUT_DIR / "cache",
}

for path in OUT.values():
    path.mkdir(parents=True, exist_ok=True)

print("Конфигурация анализа задана.")
print(f"Рабочая папка: {OUTPUT_DIR}")

In [ ]:
# @title 1.2. Настройка логирования

def setup_logger(output_dir: Path) -> logging.Logger:
    """
    Настраивает logger для текущего ноутбука.

    Parameters
    ----------
    output_dir : Path
        Папка для сохранения логов.

    Returns
    -------
    logging.Logger
        Настроенный logger.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("temporal_connectivity_events")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    log_path = output_dir / (
        "temporal_connectivity_events_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    )

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler.setFormatter(formatter)
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    logger.info("Логирование запущено.")
    logger.info("Файл лога: %s", log_path)

    return logger


logger = setup_logger(OUT["logs"])

In [ ]:
# @title 1.3. Проверка конфигурации

def validate_config(config: dict) -> None:
    """
    Проверяет корректность конфигурации.

    Parameters
    ----------
    config : dict
        Словарь конфигурации.
    """
    inventory_path = config["analysis_ready_inventory"]

    if not inventory_path.exists():
        raise FileNotFoundError(
            "Не найден analysis-ready inventory: "
            f"{inventory_path}"
        )

    if config["max_epochs_per_record"] <= 0:
        raise ValueError("max_epochs_per_record должен быть положительным.")

    if config["burst_z_threshold"] <= 0:
        raise ValueError("burst_z_threshold должен быть положительным.")

    if config["event_z_threshold"] <= 0:
        raise ValueError("event_z_threshold должен быть положительным.")

    if not config["roi_channels"]:
        raise ValueError("Словарь roi_channels не должен быть пустым.")


validate_config(CONFIG)

logger.info("CONFIG успешно проверен.")
print("CONFIG корректен. Можно переходить к загрузке данных.")

# 2. Загрузка подготовленного inventory

Загружается таблица `analysis_ready_inventory_cleaned.csv`, полученная после предобработки.

Она должна содержать:

- группу;
- идентификатор субъекта;
- идентификатор записи;
- путь к очищенному MNE Epochs-файлу;
- число эпох;
- число каналов;
- частоту дискретизации;
- длительность эпох.

In [ ]:
# @title 2.1. Загрузка analysis-ready inventory

analysis_inventory = pd.read_csv(CONFIG["analysis_ready_inventory"])

required_columns = [
    "group",
    "subject_id",
    "record_id",
    "common_epochs_path",
    "n_epochs",
    "n_common_channels",
    "sfreq_hz",
    "epoch_len_sec",
]

missing_columns = [
    column for column in required_columns
    if column not in analysis_inventory.columns
]

if missing_columns:
    raise ValueError(
        "В inventory отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

print("Inventory загружен.")
print(f"Число записей: {len(analysis_inventory)}")
print(f"Число субъектов: {analysis_inventory['subject_id'].nunique()}")

display(
    analysis_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        n_channels=("n_common_channels", "median"),
        sfreq_hz=("sfreq_hz", "median"),
        epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

In [ ]:
# @title 2.2. Проверка доступности MNE Epochs-файлов

analysis_inventory["epochs_file_exists"] = analysis_inventory[
    "common_epochs_path"
].apply(lambda path: Path(path).exists())

missing_epochs = analysis_inventory[
    ~analysis_inventory["epochs_file_exists"]
].copy()

print("Проверка путей к MNE Epochs-файлам")
print("-" * 70)
print(f"Всего записей: {len(analysis_inventory)}")
print(
    "Файлы найдены: "
    f"{analysis_inventory['epochs_file_exists'].sum()}"
)
print(f"Файлы не найдены: {len(missing_epochs)}")

if len(missing_epochs) > 0:
    display(
        missing_epochs[
            [
                "group",
                "subject_id",
                "record_id",
                "common_epochs_path",
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        "Часть MNE Epochs-файлов не найдена. "
        "Проверьте пути в common_epochs_path."
    )

logger.info("Все MNE Epochs-файлы доступны.")

In [ ]:
# @title 2.3. Визуальная сводка входных данных

summary_for_plot = (
    analysis_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
    )
    .reset_index()
)

summary_for_plot["group_ru"] = summary_for_plot["group"].map(
    GROUP_LABELS_RU
)
summary_for_plot["color"] = summary_for_plot["group"].map(GROUP_COLORS)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("n_subjects", "Число субъектов"),
    ("n_records", "Число записей"),
    ("n_epochs", "Число эпох"),
]

for ax, (column, title) in zip(axes, metrics):
    bars = ax.bar(
        summary_for_plot["group_ru"],
        summary_for_plot[column],
        color=summary_for_plot["color"],
    )

    ax.set_title(title)
    ax.set_ylabel("Количество")

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{int(height)}",
            ha="center",
            va="bottom",
        )

fig.suptitle(
    "Состав данных на входе временного и сетевого анализа",
    fontsize=17,
    y=1.03,
)

fig.tight_layout()

figure_path = OUT["figures"] / "input_inventory_summary.png"
save_figure(fig, figure_path)

plt.show()

# 3. Общие функции загрузки и агрегации

Перед расчётом признаков задаются функции:

1. загрузка MNE Epochs;
2. подвыборка эпох;
3. выбор доступных каналов ROI;
4. получение ROI-сигналов.

In [ ]:
# @title 3.1. Загрузка Epochs и подвыборка эпох

def load_epochs_file(epochs_path: str | Path) -> mne.Epochs:
    """
    Загружает MNE Epochs-файл.

    Parameters
    ----------
    epochs_path : str | Path
        Путь к FIF-файлу с эпохами.

    Returns
    -------
    mne.Epochs
        Загруженный объект MNE Epochs.
    """
    epochs_path = Path(epochs_path)

    if not epochs_path.exists():
        raise FileNotFoundError(f"Файл эпох не найден: {epochs_path}")

    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False,
    )

    return epochs


def select_epochs_subset(
    epochs: mne.Epochs,
    max_epochs: int,
    random_state: int,
) -> mne.Epochs:
    """
    Выбирает подвыборку эпох фиксированного максимального размера.

    Parameters
    ----------
    epochs : mne.Epochs
        Исходные эпохи.
    max_epochs : int
        Максимальное число эпох.
    random_state : int
        Seed генератора случайных чисел.

    Returns
    -------
    mne.Epochs
        Epochs после подвыборки.
    """
    if len(epochs) <= max_epochs:
        return epochs.copy()

    rng = np.random.default_rng(random_state)

    selected_indices = rng.choice(
        np.arange(len(epochs)),
        size=max_epochs,
        replace=False,
    )

    selected_indices = np.sort(selected_indices)

    return epochs[selected_indices].copy()


def get_roi_data(
    epochs: mne.Epochs,
    roi_channels: dict[str, list[str]],
) -> dict[str, np.ndarray]:
    """
    Возвращает усреднённый сигнал ROI.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохированные данные.
    roi_channels : dict[str, list[str]]
        Словарь ROI и каналов.

    Returns
    -------
    dict[str, np.ndarray]
        ROI-сигналы shape = (n_epochs, n_times).
    """
    data = epochs.get_data(copy=True)
    ch_to_idx = {
        channel: index for index, channel in enumerate(epochs.ch_names)
    }

    roi_data = {}

    for roi_name, channels in roi_channels.items():
        indices = [
            ch_to_idx[channel]
            for channel in channels
            if channel in ch_to_idx
        ]

        if len(indices) == 0:
            continue

        roi_data[roi_name] = np.nanmean(data[:, indices, :], axis=1)

    return roi_data

In [ ]:
# @title 3.2. Служебные функции кэширования расчётных таблиц

def load_cached_table(
    table_path: Path,
    table_name: str,
    use_cache: bool | None = None,
) -> pd.DataFrame | None:
    """
    Считывает расчётную таблицу с диска, если она уже существует.

    Parameters
    ----------
    table_path : Path
        Путь к CSV-файлу.
    table_name : str
        Название таблицы для сообщения в выводе.
    use_cache : bool | None
        Флаг использования кэша. Если None, берётся CONFIG["use_cached_tables"].

    Returns
    -------
    pd.DataFrame | None
        Таблица, если файл найден и кэширование включено; иначе None.
    """
    table_path = Path(table_path)

    if use_cache is None:
        use_cache = CONFIG.get("use_cached_tables", True)

    if use_cache and table_path.exists():
        print(f"{table_name}: найден готовый файл, считываю с диска.")
        print(f"Путь: {table_path}")
        return pd.read_csv(table_path)

    print(f"{table_name}: готовый файл не найден или кэш отключён, выполняю расчёт.")
    return None


def save_computed_table(
    df: pd.DataFrame,
    table_path: Path,
    table_name: str,
) -> pd.DataFrame:
    """
    Сохраняет расчётную таблицу и возвращает её.
    """
    save_table(df, table_path)
    print(f"{table_name}: сохранено строк: {len(df)}")
    return df


In [ ]:
# @title 3.3. Тестовая загрузка одной записи

test_row = analysis_inventory.iloc[0]

print("Тестовая запись")
print("-" * 70)
print(f"Группа: {test_row['group']}")
print(f"Субъект: {test_row['subject_id']}")
print(f"Запись: {test_row['record_id']}")

test_epochs = load_epochs_file(test_row["common_epochs_path"])

test_epochs_subset = select_epochs_subset(
    epochs=test_epochs,
    max_epochs=CONFIG["max_epochs_per_record"],
    random_state=CONFIG["epoch_random_state"],
)

test_roi_data = get_roi_data(
    epochs=test_epochs_subset,
    roi_channels=CONFIG["roi_channels"],
)

print("\nПараметры Epochs:")
print(f"Число эпох исходно: {len(test_epochs)}")
print(f"Число эпох после подвыборки: {len(test_epochs_subset)}")
print(f"Число каналов: {len(test_epochs_subset.ch_names)}")
print(f"Частота дискретизации: {test_epochs_subset.info['sfreq']} Гц")

print("\nДоступные ROI:")
for roi_name, roi_signal in test_roi_data.items():
    print(f"{roi_name}: {roi_signal.shape}")

del test_epochs
del test_epochs_subset
gc.collect()

# 4. DFA / LRTC

DFA используется для оценки долговременной временной структуры амплитудной огибающей ритмов.

Для каждой записи, ROI и диапазона:

1. сигнал фильтруется в заданном диапазоне;
2. рассчитывается амплитудная огибающая через преобразование Гильберта;
3. огибающая объединяется по эпохам;
4. рассчитывается DFA-экспонента.

DFA-экспонента описывает степень долговременной автокоррелированности амплитудной динамики.

In [ ]:
# @title 4.1. Функции DFA

def compute_envelope(
    signal: np.ndarray,
    sfreq: float,
    band: tuple[float, float],
) -> np.ndarray:
    """
    Рассчитывает амплитудную огибающую сигнала.

    Parameters
    ----------
    signal : np.ndarray
        Сигнал shape = (n_epochs, n_times).
    sfreq : float
        Частота дискретизации.
    band : tuple[float, float]
        Частотный диапазон.

    Returns
    -------
    np.ndarray
        Амплитудная огибающая shape = (n_samples,).
    """
    band_min, band_max = band

    filtered = mne.filter.filter_data(
        data=signal,
        sfreq=sfreq,
        l_freq=band_min,
        h_freq=band_max,
        verbose=False,
    )

    analytic = spsig.hilbert(filtered, axis=-1)
    envelope = np.abs(analytic)

    return envelope.reshape(-1)


def dfa_exponent(
    x: np.ndarray,
    sfreq: float,
    min_window_sec: float,
    max_window_sec: float,
    n_windows: int,
) -> float:
    """
    Рассчитывает DFA-экспоненту.

    Parameters
    ----------
    x : np.ndarray
        Одномерный сигнал.
    sfreq : float
        Частота дискретизации.
    min_window_sec : float
        Минимальный размер окна.
    max_window_sec : float
        Максимальный размер окна.
    n_windows : int
        Число размеров окон.

    Returns
    -------
    float
        DFA-экспонента.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < sfreq * max_window_sec:
        return np.nan

    x = x - np.mean(x)
    y = np.cumsum(x)

    window_sizes = np.unique(
        np.logspace(
            np.log10(min_window_sec * sfreq),
            np.log10(max_window_sec * sfreq),
            n_windows,
        ).astype(int)
    )

    fluctuations = []
    valid_windows = []

    for window_size in window_sizes:
        if window_size < 4:
            continue

        n_segments = len(y) // window_size

        if n_segments < 2:
            continue

        y_cut = y[: n_segments * window_size]
        segments = y_cut.reshape(n_segments, window_size)

        rms_values = []

        time_index = np.arange(window_size)

        for segment in segments:
            coeffs = np.polyfit(time_index, segment, deg=1)
            trend = np.polyval(coeffs, time_index)
            rms_values.append(np.sqrt(np.mean((segment - trend) ** 2)))

        fluctuations.append(np.mean(rms_values))
        valid_windows.append(window_size)

    if len(fluctuations) < 3:
        return np.nan

    slope, _ = np.polyfit(
        np.log(valid_windows),
        np.log(fluctuations),
        deg=1,
    )

    return float(slope)

In [ ]:
# @title 4.2. Массовый расчёт DFA

dfa_path = OUT["tables"] / "dfa_features_long.csv"
dfa_features_long = load_cached_table(
    dfa_path,
    table_name="DFA features",
)

if dfa_features_long is None:
    dfa_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расчёт DFA",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])
            roi_data = get_roi_data(epochs, CONFIG["roi_channels"])

            for roi_name, roi_signal in roi_data.items():
                for band_name, band_range in CONFIG["temporal_bands"].items():
                    envelope = compute_envelope(
                        signal=roi_signal,
                        sfreq=sfreq,
                        band=band_range,
                    )

                    exponent = dfa_exponent(
                        x=envelope,
                        sfreq=sfreq,
                        min_window_sec=CONFIG["dfa_min_window_sec"],
                        max_window_sec=CONFIG["dfa_max_window_sec"],
                        n_windows=CONFIG["dfa_n_windows"],
                    )

                    dfa_records.append(
                        {
                            "group": row["group"],
                            "subject_id": row["subject_id"],
                            "record_id": row["record_id"],
                            "roi": roi_name,
                            "band": band_name,
                            "dfa_exponent": exponent,
                        }
                    )

            del epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка DFA для записи %s: %s",
                row.get("record_id"),
                error,
            )

    dfa_features_long = pd.DataFrame(dfa_records)
    dfa_features_long = save_computed_table(
        dfa_features_long,
        dfa_path,
        table_name="DFA features",
    )

print("DFA features shape:", dfa_features_long.shape)
display(dfa_features_long.head())


In [ ]:
# @title 4.3. LRTC/DFA амплитудных огибающих по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

band_order = ["alpha", "beta"]

band_titles = {
    "alpha": "A. DFA exponent — α-огибающая",
    "beta": "B. DFA exponent — β-огибающая",
}

# Если эти цвета уже заданы в ячейке стиля, можно этот блок не дублировать.
COL_TBI = "#A6192E"      # тёмно-красный
COL_CONTROL = "#1F497D"  # тёмно-синий


# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = dfa_features_long.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

required_columns = [
    "group_norm",
    "subject_id",
    "record_id",
    "roi",
    "band",
    "dfa_exponent",
]

missing_columns = [
    col for col in required_columns
    if col not in plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В dfa_features_long отсутствуют нужные колонки: "
        f"{missing_columns}"
    )


# ---------------------------------------------------------------------
# Subject-level агрегация для финальной визуализации
# ---------------------------------------------------------------------
# Если у одного субъекта несколько записей, сначала усредняем внутри субъекта.
# Так график соответствует корректному subject-level уровню интерпретации.
plot_subject = (
    plot_df
    .groupby(["group_norm", "subject_id", "roi", "band"], as_index=False)
    .agg(
        dfa_exponent=("dfa_exponent", "mean"),
    )
)

print("Record-level:", plot_df.shape)
print("Subject-level:", plot_subject.shape)

display(
    plot_subject
    .groupby("group_norm")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)


# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.2),
    sharey=True,
)

fig.suptitle(
    "LRTC (DFA) амплитудной огибающей: сравнение групп по ROI",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for ax, band_name in zip(axes, band_order):
    band_df = plot_subject[plot_subject["band"] == band_name].copy()

    tbi_data = [
        band_df.loc[
            (band_df["group_norm"] == "tbi")
            & (band_df["roi"] == roi_name),
            "dfa_exponent",
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    control_data = [
        band_df.loc[
            (band_df["group_norm"] == "control")
            & (band_df["roi"] == roi_name),
            "dfa_exponent",
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.6,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.6,
            "alpha": 0.55,
        },
    )

    ax.axhline(
        0.5,
        color="gray",
        linestyle="--",
        linewidth=1.0,
        alpha=0.7,
    )

    ax.text(
        0.98,
        0.07,
        "DFA = 0.5",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
        color="dimgray",
    )

    ax.set_title(
        band_titles.get(band_name, band_name),
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.set_xticks(roi_positions)
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=22,
        ha="right",
    )

    ax.set_xlabel("Область ROI")
    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

axes[0].set_ylabel("DFA-экспонента (LRTC)")

# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.subplots_adjust(
    left=0.07,
    right=0.98,
    top=0.84,
    bottom=0.23,
    wspace=0.12,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "dfa_lrtc_envelope_by_roi_subject_level.png"

save_figure(fig, figure_path)

plt.show()

# 5. Burst-анализ

Burst-анализ описывает транзиентную организацию амплитудной динамики ритмов.

Для каждой записи, ROI и диапазона рассчитываются:

- burst rate;
- доля времени в burst;
- средняя длительность burst;
- медианная длительность burst;
- средняя пиковая амплитуда burst;
- средний меж-burst интервал.

Burst определяется по z-нормированной амплитудной огибающей.

In [ ]:
# @title 5.1. Функции burst-анализа

def detect_bursts_from_envelope(
    envelope: np.ndarray,
    sfreq: float,
    z_threshold: float,
    min_duration_sec: float,
    merge_gap_sec: float,
) -> list[dict]:
    """
    Детектирует burst-эпизоды по амплитудной огибающей.

    Parameters
    ----------
    envelope : np.ndarray
        Амплитудная огибающая.
    sfreq : float
        Частота дискретизации.
    z_threshold : float
        Z-порог.
    min_duration_sec : float
        Минимальная длительность burst.
    merge_gap_sec : float
        Максимальный разрыв для объединения соседних burst.

    Returns
    -------
    list[dict]
        Список burst-эпизодов.
    """
    envelope = np.asarray(envelope, dtype=float)
    envelope = envelope[np.isfinite(envelope)]

    if len(envelope) == 0 or np.nanstd(envelope) == 0:
        return []

    z_envelope = (envelope - np.nanmean(envelope)) / np.nanstd(envelope)
    active = z_envelope >= z_threshold

    changes = np.diff(active.astype(int), prepend=0, append=0)
    starts = np.where(changes == 1)[0]
    stops = np.where(changes == -1)[0]

    min_samples = int(min_duration_sec * sfreq)
    merge_gap_samples = int(merge_gap_sec * sfreq)

    intervals = []

    for start, stop in zip(starts, stops):
        if stop - start >= min_samples:
            intervals.append([start, stop])

    if not intervals:
        return []

    merged = [intervals[0]]

    for start, stop in intervals[1:]:
        previous_start, previous_stop = merged[-1]

        if start - previous_stop <= merge_gap_samples:
            merged[-1][1] = stop
        else:
            merged.append([start, stop])

    bursts = []

    for start, stop in merged:
        segment = z_envelope[start:stop]

        bursts.append(
            {
                "start_sample": int(start),
                "stop_sample": int(stop),
                "duration_sec": float((stop - start) / sfreq),
                "peak_z": float(np.nanmax(segment)),
                "mean_z": float(np.nanmean(segment)),
            }
        )

    return bursts


def summarize_bursts(
    bursts: list[dict],
    total_duration_sec: float,
) -> dict:
    """
    Агрегирует burst-эпизоды в признаки записи.

    Parameters
    ----------
    bursts : list[dict]
        Список burst.
    total_duration_sec : float
        Общая длительность сигнала.

    Returns
    -------
    dict
        Burst-признаки.
    """
    if not bursts or total_duration_sec <= 0:
        return {
            "burst_rate_per_min": 0.0,
            "burst_time_fraction": 0.0,
            "burst_mean_duration_sec": np.nan,
            "burst_median_duration_sec": np.nan,
            "burst_mean_peak_z": np.nan,
            "burst_mean_interburst_interval_sec": np.nan,
        }

    durations = np.array([burst["duration_sec"] for burst in bursts])
    peaks = np.array([burst["peak_z"] for burst in bursts])

    starts = np.array([burst["start_sample"] for burst in bursts])
    stops = np.array([burst["stop_sample"] for burst in bursts])

    if len(bursts) > 1:
        interburst_intervals = (starts[1:] - stops[:-1])
        # Перевод в секунды выполняется позже через duration ratio.
        mean_ibi_sec = np.nan
    else:
        mean_ibi_sec = np.nan

    return {
        "burst_rate_per_min": float(len(bursts) / total_duration_sec * 60),
        "burst_time_fraction": float(np.sum(durations) / total_duration_sec),
        "burst_mean_duration_sec": float(np.nanmean(durations)),
        "burst_median_duration_sec": float(np.nanmedian(durations)),
        "burst_mean_peak_z": float(np.nanmean(peaks)),
        "burst_mean_interburst_interval_sec": mean_ibi_sec,
    }

In [ ]:
# @title 5.2. Массовый расчёт burst-признаков

burst_path = OUT["tables"] / "burst_features_long.csv"
burst_features_long = load_cached_table(
    burst_path,
    table_name="Burst features",
)

if burst_features_long is None:
    burst_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расчёт burst-признаков",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])
            roi_data = get_roi_data(epochs, CONFIG["roi_channels"])

            for roi_name, roi_signal in roi_data.items():
                for band_name, band_range in CONFIG["temporal_bands"].items():
                    envelope = compute_envelope(
                        signal=roi_signal,
                        sfreq=sfreq,
                        band=band_range,
                    )

                    bursts = detect_bursts_from_envelope(
                        envelope=envelope,
                        sfreq=sfreq,
                        z_threshold=CONFIG["burst_z_threshold"],
                        min_duration_sec=CONFIG["burst_min_duration_sec"],
                        merge_gap_sec=CONFIG["burst_merge_gap_sec"],
                    )

                    total_duration_sec = len(envelope) / sfreq
                    burst_summary = summarize_bursts(
                        bursts=bursts,
                        total_duration_sec=total_duration_sec,
                    )

                    burst_records.append(
                        {
                            "group": row["group"],
                            "subject_id": row["subject_id"],
                            "record_id": row["record_id"],
                            "roi": roi_name,
                            "band": band_name,
                            "n_bursts": len(bursts),
                            **burst_summary,
                        }
                    )

            del epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка burst для записи %s: %s",
                row.get("record_id"),
                error,
            )

    burst_features_long = pd.DataFrame(burst_records)
    burst_features_long = save_computed_table(
        burst_features_long,
        burst_path,
        table_name="Burst features",
    )

print("Burst features shape:", burst_features_long.shape)
display(burst_features_long.head())


In [ ]:
# @title 5.3. Burst-метрики амплитудной огибающей по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ---------------------------------------------------------------------
# Настройки отображения
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]
band_order = ["alpha", "beta"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

band_labels = {
    "alpha": "α",
    "beta": "β",
}

# Если цвета уже заданы в общей стилевой ячейке, этот блок можно удалить.
COL_TBI = "#A6192E"      # тёмно-красный
COL_CONTROL = "#1F497D"  # тёмно-синий


# ---------------------------------------------------------------------
# Проверка входной таблицы
# ---------------------------------------------------------------------
if "burst_features_long" not in globals():
    raise NameError(
        "Не найдена таблица burst_features_long. "
        "Сначала выполните ячейку 5.2 с расчётом или загрузкой burst-признаков."
    )

plot_df = burst_features_long.copy()

required_columns = [
    "group",
    "subject_id",
    "roi",
    "band",
    "burst_rate_per_min",
    "burst_time_fraction",
    "burst_mean_duration_sec",
    "burst_mean_peak_z",
]

missing_columns = [
    column for column in required_columns
    if column not in plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В burst_features_long отсутствуют нужные колонки: "
        f"{missing_columns}\n\n"
        f"Доступные колонки:\n{list(plot_df.columns)}"
    )


# ---------------------------------------------------------------------
# Нормализация групп и подготовка display-метрик
# ---------------------------------------------------------------------
plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

# Доля времени в burst переводится в проценты только для графика.
plot_df["burst_time_percent"] = plot_df["burst_time_fraction"] * 100


# ---------------------------------------------------------------------
# Subject-level агрегация
# ---------------------------------------------------------------------
# Если у одного субъекта несколько записей, усредняем признаки внутри
# субъекта отдельно для каждой ROI и каждого диапазона.
metric_columns = [
    "burst_rate_per_min",
    "burst_time_percent",
    "burst_mean_duration_sec",
    "burst_mean_peak_z",
]

plot_subject = (
    plot_df
    .replace([np.inf, -np.inf], np.nan)
    .groupby(["group_norm", "subject_id", "roi", "band"], as_index=False)
    .agg(
        {
            column: "mean"
            for column in metric_columns
        }
    )
)

print("Record-level:", plot_df.shape)
print("Subject-level:", plot_subject.shape)

display(
    plot_subject
    .groupby("group_norm")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)


# ---------------------------------------------------------------------
# Метрики для 4 горизонтальных колонок
# ---------------------------------------------------------------------
metrics_to_plot = [
    {
        "column": "burst_rate_per_min",
        "title": "Частота burst",
        "ylabel": "Burst / мин",
    },
    {
        "column": "burst_time_percent",
        "title": "Доля времени в burst",
        "ylabel": "Время в burst, %",
    },
    {
        "column": "burst_mean_duration_sec",
        "title": "Средняя длительность burst",
        "ylabel": "Длительность, с",
    },
    {
        "column": "burst_mean_peak_z",
        "title": "Средняя пиковая амплитуда",
        "ylabel": "Пиковая амплитуда, z",
    },
]


# ---------------------------------------------------------------------
# Функция boxplot в едином стиле
# ---------------------------------------------------------------------
def draw_group_boxplot(
    ax,
    data,
    positions,
    color,
    width=0.32,
    showfliers=True,
):
    """
    Рисует boxplot для одной группы в едином стиле.
    """
    ax.boxplot(
        data,
        positions=positions,
        widths=width,
        patch_artist=True,
        showfliers=showfliers,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": color,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": color,
            "markeredgecolor": color,
            "markersize": 2.3,
            "alpha": 0.45,
        },
    )


# ---------------------------------------------------------------------
# Фигура: 2 строки × 4 колонки
# Верхняя строка — alpha, нижняя строка — beta.
# ---------------------------------------------------------------------
n_rows = len(band_order)
n_cols = len(metrics_to_plot)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 8.2),
    sharex=False,
    sharey=False,
)

axes = np.asarray(axes).reshape(n_rows, n_cols)

fig.suptitle(
    "Burst-метрики амплитудной огибающей (α/β): сравнение групп по ROI",
    fontsize=14,
    fontweight="bold",
    y=0.985,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

# Можно поставить False, если отрисовка идёт слишком долго.
SHOW_FLIERS = True

for row_idx, band_name in enumerate(band_order):
    band_df = plot_subject[plot_subject["band"] == band_name].copy()

    for col_idx, metric in enumerate(metrics_to_plot):
        ax = axes[row_idx, col_idx]
        feature_col = metric["column"]

        tbi_data = [
            band_df.loc[
                (band_df["group_norm"] == "tbi")
                & (band_df["roi"] == roi_name),
                feature_col,
            ]
            .dropna()
            .to_numpy()
            for roi_name in roi_order
        ]

        control_data = [
            band_df.loc[
                (band_df["group_norm"] == "control")
                & (band_df["roi"] == roi_name),
                feature_col,
            ]
            .dropna()
            .to_numpy()
            for roi_name in roi_order
        ]

        draw_group_boxplot(
            ax=ax,
            data=tbi_data,
            positions=roi_positions - offset,
            color=COL_TBI,
            width=width,
            showfliers=SHOW_FLIERS,
        )

        draw_group_boxplot(
            ax=ax,
            data=control_data,
            positions=roi_positions + offset,
            color=COL_CONTROL,
            width=width,
            showfliers=SHOW_FLIERS,
        )

        ax.set_title(
            f"{band_labels[band_name]}: {metric['title']}",
            fontsize=11,
            fontweight="bold",
            pad=6,
        )

        ax.set_ylabel(metric["ylabel"], fontsize=10)

        ax.set_xticks(roi_positions)
        ax.set_xticklabels(
            [roi_labels[roi] for roi in roi_order],
            rotation=22,
            ha="right",
            fontsize=9,
        )

        if row_idx == n_rows - 1:
            ax.set_xlabel("Область ROI", fontsize=10)

        ax.grid(axis="y", linestyle="--", alpha=0.22)
        ax.grid(axis="x", linestyle=":", alpha=0.10)

        # -------------------------------------------------------------
        # Robust-ограничение оси Y.
        # Данные не удаляются, ограничивается только отображение.
        # -------------------------------------------------------------
        values = (
            band_df[feature_col]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .to_numpy()
        )

        if len(values) > 0:
            y_low = np.nanpercentile(values, 0.5)
            y_high = np.nanpercentile(values, 99.2)
            y_range = y_high - y_low

            if np.isfinite(y_range) and y_range > 0:
                lower = y_low - 0.05 * y_range
                upper = y_high + 0.10 * y_range

                if y_low >= 0:
                    lower = max(0, lower)

                ax.set_ylim(lower, upper)


# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.012),
)

fig.subplots_adjust(
    left=0.055,
    right=0.99,
    top=0.90,
    bottom=0.145,
    wspace=0.25,
    hspace=0.42,
)


# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "burst_metrics_envelope_by_roi_subject_level_horizontal.png"

save_figure(fig, figure_path)

plt.show()

# 6. Функциональная связность и сетевые метрики

В этом разделе рассчитываются признаки функциональной связности между ROI.

Для каждой записи и частотного диапазона:

1. сигнал усредняется внутри ROI;
2. рассчитываются матрицы связности между ROI;
3. из матриц извлекаются глобальные сетевые признаки.

Включены методы:

- coherence;
- PLV;
- imaginary coherence;
- wPLI.

In [ ]:
# @title 6.1. Функции функциональной связности

def bandpass_roi_signal(
    signal: np.ndarray,
    sfreq: float,
    band: tuple[float, float],
) -> np.ndarray:
    """
    Фильтрует ROI-сигнал в заданном диапазоне.

    Parameters
    ----------
    signal : np.ndarray
        ROI-сигнал shape = (n_epochs, n_times).
    sfreq : float
        Частота дискретизации.
    band : tuple[float, float]
        Частотный диапазон.

    Returns
    -------
    np.ndarray
        Отфильтрованный сигнал shape = (n_samples,).
    """
    filtered = mne.filter.filter_data(
        data=signal,
        sfreq=sfreq,
        l_freq=band[0],
        h_freq=band[1],
        verbose=False,
    )

    return filtered.reshape(-1)


def compute_pairwise_connectivity(
    roi_series: dict[str, np.ndarray],
    sfreq: float,
) -> dict[str, np.ndarray]:
    """
    Рассчитывает матрицы связности между ROI.

    Parameters
    ----------
    roi_series : dict[str, np.ndarray]
        Словарь ROI-сигналов.
    sfreq : float
        Частота дискретизации.

    Returns
    -------
    dict[str, np.ndarray]
        Матрицы связности.
    """
    roi_names = list(roi_series.keys())
    n_roi = len(roi_names)

    matrices = {
        "coh": np.eye(n_roi),
        "plv": np.eye(n_roi),
        "imcoh": np.eye(n_roi),
        "wpli": np.eye(n_roi),
    }

    analytic = {
        roi: spsig.hilbert(signal)
        for roi, signal in roi_series.items()
    }

    phases = {
        roi: np.angle(value)
        for roi, value in analytic.items()
    }

    for i in range(n_roi):
        for j in range(i + 1, n_roi):
            roi_i = roi_names[i]
            roi_j = roi_names[j]

            sig_i = roi_series[roi_i]
            sig_j = roi_series[roi_j]

            # Coherence усредняется по частотам внутри уже отфильтрованного сигнала.
            freqs, coh_values = spsig.coherence(
                sig_i,
                sig_j,
                fs=sfreq,
                nperseg=min(int(sfreq * 2), len(sig_i)),
            )

            coh = float(np.nanmean(coh_values))

            phase_diff = phases[roi_i] - phases[roi_j]
            plv = float(np.abs(np.nanmean(np.exp(1j * phase_diff))))

            cross_spectrum_imag = np.imag(
                analytic[roi_i] * np.conj(analytic[roi_j])
            )

            imcoh = float(
                np.abs(np.nanmean(cross_spectrum_imag))
                / (
                    np.sqrt(
                        np.nanmean(np.abs(analytic[roi_i]) ** 2)
                        * np.nanmean(np.abs(analytic[roi_j]) ** 2)
                    )
                    + 1e-12
                )
            )

            wpli = float(
                np.abs(np.nanmean(cross_spectrum_imag))
                / (np.nanmean(np.abs(cross_spectrum_imag)) + 1e-12)
            )

            matrices["coh"][i, j] = coh
            matrices["coh"][j, i] = coh

            matrices["plv"][i, j] = plv
            matrices["plv"][j, i] = plv

            matrices["imcoh"][i, j] = imcoh
            matrices["imcoh"][j, i] = imcoh

            matrices["wpli"][i, j] = wpli
            matrices["wpli"][j, i] = wpli

    matrices["roi_names"] = roi_names

    return matrices


def summarize_connectivity_matrix(matrix: np.ndarray) -> dict:
    """
    Агрегирует матрицу связности в сетевые признаки.

    Parameters
    ----------
    matrix : np.ndarray
        Матрица связности.

    Returns
    -------
    dict
        Сетевые признаки.
    """
    if matrix.shape[0] < 2:
        return {
            "mean_strength": np.nan,
            "median_strength": np.nan,
            "density_75": np.nan,
            "clustering": np.nan,
        }

    upper = matrix[np.triu_indices_from(matrix, k=1)]

    threshold = np.nanpercentile(upper, 75)
    adjacency = (matrix >= threshold).astype(float)
    np.fill_diagonal(adjacency, 0)

    graph = nx.from_numpy_array(adjacency)

    return {
        "mean_strength": float(np.nanmean(upper)),
        "median_strength": float(np.nanmedian(upper)),
        "density_75": float(nx.density(graph)),
        "clustering": float(nx.average_clustering(graph)),
    }

In [ ]:
# @title 6.2. Массовый расчёт функциональной связности

connectivity_path = OUT["tables"] / "connectivity_features_long.csv"
connectivity_features_long = load_cached_table(
    connectivity_path,
    table_name="Connectivity features",
)

if connectivity_features_long is None:
    connectivity_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расчёт связности",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])
            roi_data = get_roi_data(epochs, CONFIG["roi_channels"])

            for band_name, band_range in CONFIG["connectivity_bands"].items():
                roi_series = {
                    roi_name: bandpass_roi_signal(
                        signal=roi_signal,
                        sfreq=sfreq,
                        band=band_range,
                    )
                    for roi_name, roi_signal in roi_data.items()
                }

                matrices = compute_pairwise_connectivity(
                    roi_series=roi_series,
                    sfreq=sfreq,
                )

                for method_name in CONFIG["connectivity_methods"]:
                    matrix = matrices[method_name]
                    summary = summarize_connectivity_matrix(matrix)

                    connectivity_records.append(
                        {
                            "group": row["group"],
                            "subject_id": row["subject_id"],
                            "record_id": row["record_id"],
                            "band": band_name,
                            "method": method_name,
                            "n_roi_used": len(matrices["roi_names"]),
                            **summary,
                        }
                    )

            del epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка connectivity для записи %s: %s",
                row.get("record_id"),
                error,
            )

    connectivity_features_long = pd.DataFrame(connectivity_records)
    connectivity_features_long = save_computed_table(
        connectivity_features_long,
        connectivity_path,
        table_name="Connectivity features",
    )

print("Connectivity features shape:", connectivity_features_long.shape)
display(connectivity_features_long.head())


In [ ]:
# @title 6.3. Визуализация средней силы функциональной связности

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ---------------------------------------------------------------------
# Проверка входных данных
# ---------------------------------------------------------------------
if "connectivity_features_long" not in globals():
    raise NameError(
        "Не найдена таблица connectivity_features_long. "
        "Сначала выполните расчёт или загрузку признаков связности."
    )

if len(connectivity_features_long) == 0:
    raise ValueError("connectivity_features_long пустая.")

required_columns = [
    "group",
    "subject_id",
    "method",
    "band",
    "mean_strength",
]

missing_columns = [
    column for column in required_columns
    if column not in connectivity_features_long.columns
]

if missing_columns:
    raise ValueError(
        "В connectivity_features_long отсутствуют нужные колонки: "
        f"{missing_columns}\n\n"
        f"Доступные колонки:\n{list(connectivity_features_long.columns)}"
    )


# ---------------------------------------------------------------------
# Настройки отображения
# ---------------------------------------------------------------------
COL_TBI = "#A6192E"
COL_CONTROL = "#1F497D"

group_order = ["tbi", "control"]

group_labels = {
    "tbi": "ЧМТ",
    "control": "Контроль",
}

group_colors = {
    "tbi": COL_TBI,
    "control": COL_CONTROL,
}

band_labels = {
    "delta": "δ",
    "theta": "θ",
    "alpha": "α",
    "beta": "β",
    "gamma": "γ",
}

method_labels = {
    "coh": "COH",
    "coherence": "COH",
    "plv": "PLV",
    "wpli": "wPLI",
    "pli": "PLI",
    "imcoh": "ImCoh",
}


# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = connectivity_features_long.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(group_order)].copy()

plot_df = (
    plot_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["mean_strength"])
    .copy()
)


# ---------------------------------------------------------------------
# Subject-level агрегация
# ---------------------------------------------------------------------
# Если у субъекта несколько записей, сначала усредняем mean_strength
# внутри субъекта отдельно для каждого method × band.
# ---------------------------------------------------------------------
conn_subject = (
    plot_df
    .groupby(["group_norm", "subject_id", "method", "band"], as_index=False)
    .agg(
        mean_strength=("mean_strength", "mean"),
    )
)

print("Record-level:", plot_df.shape)
print("Subject-level:", conn_subject.shape)

display(
    conn_subject
    .groupby("group_norm")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)


# ---------------------------------------------------------------------
# Сводка mean ± SEM по субъектам
# ---------------------------------------------------------------------
summary = (
    conn_subject
    .groupby(["group_norm", "method", "band"], as_index=False)
    .agg(
        mean_strength=("mean_strength", "mean"),
        sem_strength=("mean_strength", "sem"),
        n_subjects=("mean_strength", "count"),
    )
)

if "connectivity_methods" in CONFIG:
    methods = [
        method for method in CONFIG["connectivity_methods"]
        if method in summary["method"].unique()
    ]
else:
    methods = sorted(summary["method"].dropna().unique())

if "connectivity_bands" in CONFIG:
    bands = [
        band for band in CONFIG["connectivity_bands"]
        if band in summary["band"].unique()
    ]
elif "temporal_bands" in CONFIG:
    bands = [
        band for band in CONFIG["temporal_bands"].keys()
        if band in summary["band"].unique()
    ]
else:
    bands = sorted(summary["band"].dropna().unique())

if not methods:
    raise ValueError("Не найдено методов связности для визуализации.")

if not bands:
    raise ValueError("Не найдено частотных диапазонов для визуализации.")


# ---------------------------------------------------------------------
# Фигура: одна панель на метод, внутри панели band × group
# ---------------------------------------------------------------------
n_methods = len(methods)

fig, axes = plt.subplots(
    1,
    n_methods,
    figsize=(5.0 * n_methods, 4.8),
    sharey=True,
)

axes = np.atleast_1d(axes)

fig.suptitle(
    "Средняя сила функциональной связности: сравнение групп",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

x = np.arange(len(bands))
width = 0.34

for ax, method in zip(axes, methods):
    method_df = summary[summary["method"] == method].copy()

    for offset, group_norm in zip([-width / 2, width / 2], group_order):
        group_df = method_df[method_df["group_norm"] == group_norm]

        values = []
        errors = []

        for band in bands:
            sub = group_df[group_df["band"] == band]

            if sub.empty:
                values.append(np.nan)
                errors.append(0.0)
            else:
                values.append(float(sub["mean_strength"].iloc[0]))
                errors.append(float(sub["sem_strength"].iloc[0]))

        ax.bar(
            x + offset,
            values,
            width=width,
            yerr=errors,
            color=group_colors[group_norm],
            edgecolor="black",
            linewidth=0.8,
            capsize=4,
            label=group_labels[group_norm],
        )

    ax.set_title(
        method_labels.get(method, str(method).upper()),
        fontsize=12,
        fontweight="bold",
        pad=8,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [band_labels.get(band, band) for band in bands],
        fontsize=10,
    )

    ax.set_xlabel("Диапазон частот")
    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

axes[0].set_ylabel("Средняя сила связности")

# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.subplots_adjust(
    left=0.07,
    right=0.98,
    top=0.82,
    bottom=0.20,
    wspace=0.15,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "connectivity_mean_strength_by_method_band_subject_level.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 6.4. Матрицы ROI×ROI связности: ЧМТ / Контроль / разность

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ---------------------------------------------------------------------
# Настройки: что визуализируем
# ---------------------------------------------------------------------
# По умолчанию пробуем wPLI в alpha-диапазоне.
# Если wPLI нет, можно заменить на "plv".
SELECTED_METHOD = "wpli"
SELECTED_BAND = "alpha"

# ---------------------------------------------------------------------
# Настройки названий таблицы и колонок
# ---------------------------------------------------------------------
# Укажи имя своей long-таблицы ROI×ROI связей.
# Частые варианты: connectivity_edges_long, connectivity_roi_edges_long,
# roi_connectivity_edges_long, connectivity_matrices_long.
candidate_table_names = [
    "connectivity_edges_long",
    "connectivity_roi_edges_long",
    "roi_connectivity_edges_long",
    "connectivity_matrices_long",
]

conn_edges = None
used_table_name = None

for table_name in candidate_table_names:
    if table_name in globals():
        conn_edges = globals()[table_name].copy()
        used_table_name = table_name
        break

if conn_edges is None:
    raise NameError(
        "Не найдена long-таблица ROI×ROI связности. "
        "Ожидается одна из таблиц: "
        f"{candidate_table_names}. "
        "Нужны колонки group, subject_id, method, band, roi_i, roi_j, value."
    )

print(f"Используется таблица связности: {used_table_name}")

# ---------------------------------------------------------------------
# Если названия колонок отличаются, поменяй их здесь
# ---------------------------------------------------------------------
ROI_1_COL = "roi_i"
ROI_2_COL = "roi_j"
VALUE_COL = "connectivity_value"
METHOD_COL = "method"
BAND_COL = "band"

# Автоподбор альтернативных названий
column_aliases = {
    "roi_i": ["roi_i", "roi_1", "source_roi", "roi_a", "region_i"],
    "roi_j": ["roi_j", "roi_2", "target_roi", "roi_b", "region_j"],
    "connectivity_value": [
        "connectivity_value",
        "value",
        "conn_value",
        "connectivity",
        "mean_connectivity",
        "edge_value",
    ],
}

def find_existing_column(df, candidates):
    """Возвращает первое найденное имя колонки из списка candidates."""
    for column in candidates:
        if column in df.columns:
            return column
    return None


ROI_1_COL = find_existing_column(conn_edges, column_aliases["roi_i"])
ROI_2_COL = find_existing_column(conn_edges, column_aliases["roi_j"])
VALUE_COL = find_existing_column(conn_edges, column_aliases["connectivity_value"])

required_columns = [
    "group",
    "subject_id",
    METHOD_COL,
    BAND_COL,
    ROI_1_COL,
    ROI_2_COL,
    VALUE_COL,
]

missing_columns = [
    column for column in required_columns
    if column is None or column not in conn_edges.columns
]

if missing_columns:
    raise ValueError(
        "В таблице связности не найдены нужные колонки.\n"
        f"Доступные колонки:\n{list(conn_edges.columns)}\n\n"
        "Проверь ROI_1_COL, ROI_2_COL и VALUE_COL."
    )


# ---------------------------------------------------------------------
# Подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

band_labels = {
    "delta": "δ",
    "theta": "θ",
    "alpha": "α",
    "beta": "β",
    "gamma": "γ",
}

method_labels = {
    "coh": "COH",
    "coherence": "COH",
    "plv": "PLV",
    "wpli": "wPLI",
    "pli": "PLI",
    "imcoh": "ImCoh",
}


# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
conn_edges["group_norm"] = conn_edges["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

conn_edges = conn_edges[conn_edges["group_norm"].isin(["tbi", "control"])].copy()

conn_edges = (
    conn_edges
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[VALUE_COL])
    .copy()
)

# Если выбранного метода нет, автоматически пробуем PLV.
available_methods = sorted(conn_edges[METHOD_COL].dropna().unique())
available_bands = sorted(conn_edges[BAND_COL].dropna().unique())

if SELECTED_METHOD not in available_methods:
    if "plv" in available_methods:
        print(
            f"Метод {SELECTED_METHOD} не найден. "
            "Автоматически выбран PLV."
        )
        SELECTED_METHOD = "plv"
    else:
        SELECTED_METHOD = available_methods[0]
        print(f"Автоматически выбран метод: {SELECTED_METHOD}")

if SELECTED_BAND not in available_bands:
    SELECTED_BAND = available_bands[0]
    print(f"Автоматически выбран диапазон: {SELECTED_BAND}")


# ---------------------------------------------------------------------
# Subject-level агрегация
# ---------------------------------------------------------------------
conn_subject_edges = (
    conn_edges
    .groupby(
        [
            "group_norm",
            "subject_id",
            METHOD_COL,
            BAND_COL,
            ROI_1_COL,
            ROI_2_COL,
        ],
        as_index=False,
    )
    .agg(
        connectivity_value=(VALUE_COL, "mean"),
    )
)

print("Record-level:", conn_edges.shape)
print("Subject-level:", conn_subject_edges.shape)

display(
    conn_subject_edges
    .groupby("group_norm")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)


# ---------------------------------------------------------------------
# Функции построения матриц
# ---------------------------------------------------------------------
def make_group_connectivity_matrix(
    df: pd.DataFrame,
    method: str,
    band: str,
    group_norm: str,
) -> pd.DataFrame:
    """
    Формирует среднюю ROI×ROI матрицу связности для группы.
    """
    sub = df[
        (df[METHOD_COL] == method)
        & (df[BAND_COL] == band)
        & (df["group_norm"] == group_norm)
    ].copy()

    mean_edges = (
        sub
        .groupby([ROI_1_COL, ROI_2_COL], as_index=False)
        .agg(value=("connectivity_value", "mean"))
    )

    matrix = pd.DataFrame(
        np.nan,
        index=roi_order,
        columns=roi_order,
    )

    for _, row in mean_edges.iterrows():
        r1 = row[ROI_1_COL]
        r2 = row[ROI_2_COL]

        if r1 in roi_order and r2 in roi_order:
            matrix.loc[r1, r2] = row["value"]
            matrix.loc[r2, r1] = row["value"]

    np.fill_diagonal(matrix.values, np.nan)

    return matrix


# ---------------------------------------------------------------------
# Матрицы
# ---------------------------------------------------------------------
tbi_matrix = make_group_connectivity_matrix(
    df=conn_subject_edges,
    method=SELECTED_METHOD,
    band=SELECTED_BAND,
    group_norm="tbi",
)

control_matrix = make_group_connectivity_matrix(
    df=conn_subject_edges,
    method=SELECTED_METHOD,
    band=SELECTED_BAND,
    group_norm="control",
)

diff_matrix = tbi_matrix - control_matrix


# ---------------------------------------------------------------------
# Цветовые шкалы
# ---------------------------------------------------------------------
group_vmax = np.nanmax(
    [
        np.nanmax(tbi_matrix.values),
        np.nanmax(control_matrix.values),
    ]
)

if not np.isfinite(group_vmax) or group_vmax == 0:
    group_vmax = 1.0

diff_vmax = np.nanmax(np.abs(diff_matrix.values))

if not np.isfinite(diff_vmax) or diff_vmax == 0:
    diff_vmax = 1.0


# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(13.8, 4.8),
)

fig.suptitle(
    (
        "Функциональная связность ROI×ROI: "
        f"{method_labels.get(SELECTED_METHOD, SELECTED_METHOD)} / "
        f"{band_labels.get(SELECTED_BAND, SELECTED_BAND)}-диапазон"
    ),
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

items = [
    {
        "matrix": tbi_matrix,
        "title": "ЧМТ",
        "cmap": "Blues",
        "vmin": 0,
        "vmax": group_vmax,
        "norm": None,
        "cbar_label": "Связность",
    },
    {
        "matrix": control_matrix,
        "title": "Контроль",
        "cmap": "Blues",
        "vmin": 0,
        "vmax": group_vmax,
        "norm": None,
        "cbar_label": "Связность",
    },
    {
        "matrix": diff_matrix,
        "title": "Разность: ЧМТ − Контроль",
        "cmap": "RdBu_r",
        "vmin": None,
        "vmax": None,
        "norm": TwoSlopeNorm(
            vmin=-diff_vmax,
            vcenter=0,
            vmax=diff_vmax,
        ),
        "cbar_label": "Разность",
    },
]

for ax, item in zip(axes, items):
    values = item["matrix"].values

    im = ax.imshow(
        values,
        cmap=item["cmap"],
        vmin=item["vmin"],
        vmax=item["vmax"],
        norm=item["norm"],
    )

    ax.set_title(
        item["title"],
        fontsize=12,
        fontweight="bold",
        pad=8,
    )

    ax.set_xticks(np.arange(len(roi_order)))
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=35,
        ha="right",
    )

    ax.set_yticks(np.arange(len(roi_order)))
    ax.set_yticklabels([roi_labels[roi] for roi in roi_order])

    ax.set_xticks(np.arange(-0.5, len(roi_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(roi_order), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    scale = np.nanmax(np.abs(values))

    if not np.isfinite(scale) or scale == 0:
        scale = 1.0

    for i in range(len(roi_order)):
        for j in range(len(roi_order)):
            value = values[i, j]

            if pd.isna(value):
                continue

            text_color = "white" if abs(value) > 0.55 * scale else "black"

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8.5,
                color=text_color,
            )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label(item["cbar_label"], fontsize=9)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.84,
    bottom=0.20,
    wspace=0.35,
)

figure_path = (
    OUT["figures"]
    / f"connectivity_roi_matrix_{SELECTED_METHOD}_{SELECTED_BAND}.png"
)

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 6.5. Статистическая heatmap связности: Hedges' g ROI×ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy import stats
from statsmodels.stats.multitest import multipletests


# ---------------------------------------------------------------------
# Статистические функции
# ---------------------------------------------------------------------
def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    """
    Рассчитывает Hedges' g для двух независимых групп.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_sd = np.sqrt(
        (
            (n_x - 1) * np.var(x, ddof=1)
            + (n_y - 1) * np.var(y, ddof=1)
        )
        / (n_x + n_y - 2)
    )

    if pooled_sd == 0:
        return np.nan

    cohen_d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (n_x + n_y) - 9)

    return float(cohen_d * correction)


def q_to_stars(q_value: float) -> str:
    """Преобразует q-FDR в звёздочки."""
    if pd.isna(q_value):
        return ""

    if q_value < 0.001:
        return "***"

    if q_value < 0.01:
        return "**"

    if q_value < 0.05:
        return "*"

    return ""


# ---------------------------------------------------------------------
# Статистика по ROI×ROI связям
# ---------------------------------------------------------------------
stat_records = []

selected_edges = conn_subject_edges[
    (conn_subject_edges[METHOD_COL] == SELECTED_METHOD)
    & (conn_subject_edges[BAND_COL] == SELECTED_BAND)
].copy()

for (roi_i, roi_j), sub_df in selected_edges.groupby([ROI_1_COL, ROI_2_COL]):
    tbi_values = (
        sub_df.loc[sub_df["group_norm"] == "tbi", "connectivity_value"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=float)
    )

    control_values = (
        sub_df.loc[sub_df["group_norm"] == "control", "connectivity_value"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=float)
    )

    if len(tbi_values) < 2 or len(control_values) < 2:
        u_stat = np.nan
        p_value = np.nan
        effect = np.nan
    else:
        u_stat, p_value = stats.mannwhitneyu(
            tbi_values,
            control_values,
            alternative="two-sided",
        )
        effect = hedges_g(tbi_values, control_values)

    stat_records.append(
        {
            "method": SELECTED_METHOD,
            "band": SELECTED_BAND,
            "roi_i": roi_i,
            "roi_j": roi_j,
            "tbi_mean": np.nanmean(tbi_values),
            "control_mean": np.nanmean(control_values),
            "delta_tbi_minus_control": (
                np.nanmean(tbi_values) - np.nanmean(control_values)
            ),
            "hedges_g": effect,
            "u_stat": u_stat,
            "p_value": p_value,
            "n_tbi": len(tbi_values),
            "n_control": len(control_values),
        }
    )

connectivity_edge_stats_subject = pd.DataFrame(stat_records)

if len(connectivity_edge_stats_subject) > 0:
    connectivity_edge_stats_subject["q_fdr"] = multipletests(
        connectivity_edge_stats_subject["p_value"].fillna(1.0),
        method="fdr_bh",
    )[1]

    connectivity_edge_stats_subject["significant_fdr_0_05"] = (
        connectivity_edge_stats_subject["q_fdr"] < 0.05
    )

save_table(
    connectivity_edge_stats_subject,
    OUT["tables"] / (
        f"connectivity_edge_stats_subject_"
        f"{SELECTED_METHOD}_{SELECTED_BAND}.csv"
    ),
)

display(connectivity_edge_stats_subject)


# ---------------------------------------------------------------------
# Формирование матриц Hedges' g и q-FDR
# ---------------------------------------------------------------------
g_matrix = pd.DataFrame(
    np.nan,
    index=roi_order,
    columns=roi_order,
)

q_matrix = pd.DataFrame(
    np.nan,
    index=roi_order,
    columns=roi_order,
)

for _, row in connectivity_edge_stats_subject.iterrows():
    r1 = row["roi_i"]
    r2 = row["roi_j"]

    if r1 in roi_order and r2 in roi_order:
        g_matrix.loc[r1, r2] = row["hedges_g"]
        g_matrix.loc[r2, r1] = row["hedges_g"]

        q_matrix.loc[r1, r2] = row["q_fdr"]
        q_matrix.loc[r2, r1] = row["q_fdr"]

np.fill_diagonal(g_matrix.values, np.nan)
np.fill_diagonal(q_matrix.values, np.nan)


# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
vmax = np.nanmax(np.abs(g_matrix.values))

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

norm = TwoSlopeNorm(
    vmin=-vmax,
    vcenter=0,
    vmax=vmax,
)

fig, ax = plt.subplots(figsize=(6.4, 5.6))

im = ax.imshow(
    g_matrix.values,
    cmap="RdBu_r",
    norm=norm,
)

ax.set_title(
    (
        "Размер эффекта связности ROI×ROI\n"
        f"{method_labels.get(SELECTED_METHOD, SELECTED_METHOD)} / "
        f"{band_labels.get(SELECTED_BAND, SELECTED_BAND)}-диапазон"
    ),
    fontsize=13,
    fontweight="bold",
    pad=10,
)

ax.set_xticks(np.arange(len(roi_order)))
ax.set_xticklabels(
    [roi_labels[roi] for roi in roi_order],
    rotation=35,
    ha="right",
)

ax.set_yticks(np.arange(len(roi_order)))
ax.set_yticklabels([roi_labels[roi] for roi in roi_order])

ax.set_xticks(np.arange(-0.5, len(roi_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(roi_order), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

for i in range(len(roi_order)):
    for j in range(len(roi_order)):
        g_value = g_matrix.values[i, j]
        q_value = q_matrix.values[i, j]

        if pd.isna(g_value):
            continue

        stars = q_to_stars(q_value)
        label = f"{g_value:.2f}{stars}"

        text_color = "white" if abs(g_value) > 0.55 * vmax else "black"

        ax.text(
            j,
            i,
            label,
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold" if stars else "normal",
            color=text_color,
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Hedges' g, ЧМТ − Контроль")

fig.subplots_adjust(
    left=0.18,
    right=0.92,
    top=0.86,
    bottom=0.16,
)

figure_path = (
    OUT["figures"]
    / f"connectivity_roi_effect_hedges_g_{SELECTED_METHOD}_{SELECTED_BAND}.png"
)

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 6.6. Network graph значимых ROI-связей

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ---------------------------------------------------------------------
# Настройки
# ---------------------------------------------------------------------
TOP_N_EDGES = 6

COL_TBI = "#A6192E"
COL_CONTROL = "#1F497D"

# Позиции ROI на схематической голове
roi_positions_2d = {
    "frontal": (0.0, 1.0),
    "central": (0.0, 0.35),
    "temporal": (-0.9, 0.15),
    "parietal": (0.0, -0.35),
    "occipital": (0.0, -0.95),
}

roi_labels_short = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}


# ---------------------------------------------------------------------
# Выбираем связи
# ---------------------------------------------------------------------
plot_edges = connectivity_edge_stats_subject.copy()

plot_edges = plot_edges.dropna(subset=["hedges_g"]).copy()

significant_edges = plot_edges[
    plot_edges["q_fdr"] < 0.05
].copy()

if len(significant_edges) > 0:
    edges_to_plot = significant_edges.copy()
    edge_selection_title = "значимые после FDR"
else:
    edges_to_plot = (
        plot_edges
        .assign(abs_g=lambda df: df["hedges_g"].abs())
        .sort_values("abs_g", ascending=False)
        .head(TOP_N_EDGES)
        .copy()
    )
    edge_selection_title = f"top-{TOP_N_EDGES} по |Hedges' g|"

edges_to_plot["abs_g"] = edges_to_plot["hedges_g"].abs()

if edges_to_plot.empty:
    raise ValueError("Нет связей для построения network graph.")


# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.0, 6.5))

fig.suptitle(
    (
        "ROI-сеть межгрупповых различий связности\n"
        f"{method_labels.get(SELECTED_METHOD, SELECTED_METHOD)} / "
        f"{band_labels.get(SELECTED_BAND, SELECTED_BAND)}-диапазон, "
        f"{edge_selection_title}"
    ),
    fontsize=13,
    fontweight="bold",
    y=0.97,
)

# Схематическая голова
head_circle = plt.Circle(
    (0, 0),
    1.15,
    color="lightgray",
    fill=False,
    linewidth=1.2,
    alpha=0.8,
)

ax.add_patch(head_circle)

max_abs_g = edges_to_plot["abs_g"].max()

if not np.isfinite(max_abs_g) or max_abs_g == 0:
    max_abs_g = 1.0

# Рёбра
for _, row in edges_to_plot.iterrows():
    r1 = row["roi_i"]
    r2 = row["roi_j"]

    if r1 not in roi_positions_2d or r2 not in roi_positions_2d:
        continue

    x1, y1 = roi_positions_2d[r1]
    x2, y2 = roi_positions_2d[r2]

    effect = row["hedges_g"]
    width = 1.0 + 4.0 * abs(effect) / max_abs_g

    color = COL_TBI if effect > 0 else COL_CONTROL

    ax.plot(
        [x1, x2],
        [y1, y2],
        color=color,
        linewidth=width,
        alpha=0.75,
        solid_capstyle="round",
    )

    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2

    ax.text(
        mid_x,
        mid_y,
        f"{effect:.2f}",
        ha="center",
        va="center",
        fontsize=8.5,
        color="black",
        bbox={
            "boxstyle": "round,pad=0.15",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.75,
        },
    )

# Узлы
for roi_name, (x, y) in roi_positions_2d.items():
    ax.scatter(
        x,
        y,
        s=520,
        color="white",
        edgecolor="black",
        linewidth=1.2,
        zorder=5,
    )

    ax.text(
        x,
        y,
        roi_labels_short[roi_name],
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        zorder=6,
    )

# Легенда
legend_handles = [
    Line2D(
        [0],
        [0],
        color=COL_TBI,
        linewidth=3,
        label="Выше при ЧМТ",
    ),
    Line2D(
        [0],
        [0],
        color=COL_CONTROL,
        linewidth=3,
        label="Выше в контроле",
    ),
]

ax.legend(
    handles=legend_handles,
    loc="lower center",
    frameon=False,
    ncol=2,
    bbox_to_anchor=(0.5, -0.04),
)

ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.30, 1.30)
ax.set_aspect("equal")
ax.axis("off")

fig.subplots_adjust(
    left=0.04,
    right=0.96,
    top=0.86,
    bottom=0.12,
)

figure_path = (
    OUT["figures"]
    / f"connectivity_network_top_edges_{SELECTED_METHOD}_{SELECTED_BAND}.png"
)

save_figure(fig, figure_path)

plt.show()

# 7. Расширенный блок связности: ROI×ROI матрицы и graph metrics

In [ ]:
# @title 7.1. Сохранение ROI×ROI матриц связности

connectivity_matrices_path = OUT["tables"] / "connectivity_matrices_long.csv"
connectivity_matrices_long = load_cached_table(
    connectivity_matrices_path,
    table_name="ROI×ROI connectivity matrices",
)

if connectivity_matrices_long is None:
    connectivity_matrix_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расчёт и сохранение ROI×ROI матриц",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])
            roi_data = get_roi_data(epochs, CONFIG["roi_channels"])

            for band_name, band_range in CONFIG["connectivity_bands"].items():
                roi_series = {
                    roi_name: bandpass_roi_signal(
                        signal=roi_signal,
                        sfreq=sfreq,
                        band=band_range,
                    )
                    for roi_name, roi_signal in roi_data.items()
                }

                matrices = compute_pairwise_connectivity(
                    roi_series=roi_series,
                    sfreq=sfreq,
                )

                roi_names = matrices["roi_names"]

                for method_name in CONFIG["connectivity_methods"]:
                    matrix = matrices[method_name]

                    for i, roi_i in enumerate(roi_names):
                        for j, roi_j in enumerate(roi_names):
                            connectivity_matrix_records.append(
                                {
                                    "group": row["group"],
                                    "subject_id": row["subject_id"],
                                    "record_id": row["record_id"],
                                    "band": band_name,
                                    "method": method_name,
                                    "roi_i": roi_i,
                                    "roi_j": roi_j,
                                    "connectivity": float(matrix[i, j]),
                                }
                            )

            del epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка ROI×ROI connectivity для записи %s: %s",
                row.get("record_id"),
                error,
            )

    connectivity_matrices_long = pd.DataFrame(connectivity_matrix_records)
    connectivity_matrices_long = save_computed_table(
        connectivity_matrices_long,
        connectivity_matrices_path,
        table_name="ROI×ROI connectivity matrices",
    )

print("Connectivity matrices shape:", connectivity_matrices_long.shape)
display(connectivity_matrices_long.head())


In [ ]:
# @title 7.2. Групповые средние матрицы связности

group_mean_connectivity_matrices = (
    connectivity_matrices_long
    .groupby(["group", "band", "method", "roi_i", "roi_j"], as_index=False)
    .agg(
        mean_connectivity=("connectivity", "mean"),
        sem_connectivity=("connectivity", "sem"),
    )
)

save_table(
    group_mean_connectivity_matrices,
    OUT["tables"] / "group_mean_connectivity_matrices.csv",
)

display(group_mean_connectivity_matrices.head())

In [ ]:
# @title 7.3. Расширенные graph metrics

def summarize_graph_extended(matrix: np.ndarray) -> dict:
    """
    Рассчитывает расширенные graph metrics для матрицы связности.

    Parameters
    ----------
    matrix : np.ndarray
        Матрица связности.

    Returns
    -------
    dict
        Расширенные сетевые признаки.
    """
    if matrix.shape[0] < 3:
        return {
            "global_efficiency": np.nan,
            "local_efficiency": np.nan,
            "avg_shortest_path": np.nan,
            "small_world_proxy": np.nan,
        }

    upper = matrix[np.triu_indices_from(matrix, k=1)]
    threshold = np.nanpercentile(upper, 75)

    adjacency = (matrix >= threshold).astype(float)
    np.fill_diagonal(adjacency, 0)

    graph = nx.from_numpy_array(adjacency)

    if nx.is_connected(graph):
        avg_path = nx.average_shortest_path_length(graph)
    else:
        largest_cc = graph.subgraph(
            max(nx.connected_components(graph), key=len)
        )
        avg_path = (
            nx.average_shortest_path_length(largest_cc)
            if largest_cc.number_of_nodes() > 1
            else np.nan
        )

    clustering = nx.average_clustering(graph)
    global_efficiency = nx.global_efficiency(graph)
    local_efficiency = nx.local_efficiency(graph)

    small_world_proxy = (
        clustering / avg_path
        if avg_path and np.isfinite(avg_path) and avg_path > 0
        else np.nan
    )

    return {
        "global_efficiency": float(global_efficiency),
        "local_efficiency": float(local_efficiency),
        "avg_shortest_path": float(avg_path),
        "small_world_proxy": float(small_world_proxy),
    }



extended_graph_path = OUT["tables"] / "extended_graph_features_long.csv"
extended_graph_features_long = load_cached_table(
    extended_graph_path,
    table_name="Extended graph metrics",
)

if extended_graph_features_long is None:

    extended_graph_records = []

    for (group, subject_id, record_id, band, method), sub_df in (
        connectivity_matrices_long
        .groupby(["group", "subject_id", "record_id", "band", "method"])
    ):
        roi_names = sorted(sub_df["roi_i"].unique())
        roi_to_idx = {roi: idx for idx, roi in enumerate(roi_names)}
        matrix = np.zeros((len(roi_names), len(roi_names)))

        for _, edge in sub_df.iterrows():
            i = roi_to_idx[edge["roi_i"]]
            j = roi_to_idx[edge["roi_j"]]
            matrix[i, j] = edge["connectivity"]

        extended = summarize_graph_extended(matrix)

        extended_graph_records.append(
            {
                "group": group,
                "subject_id": subject_id,
                "record_id": record_id,
                "band": band,
                "method": method,
                **extended,
            }
        )

    extended_graph_features_long = pd.DataFrame(extended_graph_records)

    extended_graph_features_long = save_computed_table(
        extended_graph_features_long,
        extended_graph_path,
        table_name="Extended graph metrics",
    )
else:
    print("Extended graph metrics загружены из кэша.")

print("Extended graph features shape:", extended_graph_features_long.shape)
display(extended_graph_features_long.head())


# 8. Событийные признаки

In [ ]:
# @title 8.1. Функции событийного анализа

def compute_line_length(signal: np.ndarray) -> float:
    """
    Рассчитывает line length сигнала.

    Parameters
    ----------
    signal : np.ndarray
        Одномерный сигнал.

    Returns
    -------
    float
        Line length.
    """
    return float(np.sum(np.abs(np.diff(signal))))


def detect_events_from_global_signal(
    data: np.ndarray,
    sfreq: float,
    z_threshold: float,
    min_duration_sec: float,
    merge_gap_sec: float,
) -> list[dict]:
    """
    Детектирует события по глобальной амплитуде.

    Parameters
    ----------
    data : np.ndarray
        Данные shape = (n_epochs, n_channels, n_times).
    sfreq : float
        Частота дискретизации.
    z_threshold : float
        Z-порог.
    min_duration_sec : float
        Минимальная длительность события.
    merge_gap_sec : float
        Максимальный разрыв для объединения событий.

    Returns
    -------
    list[dict]
        Список событий.
    """
    global_signal = np.nanmean(np.abs(data), axis=1).reshape(-1)

    if np.nanstd(global_signal) == 0:
        return []

    z_signal = (
        global_signal - np.nanmean(global_signal)
    ) / np.nanstd(global_signal)

    active = z_signal >= z_threshold
    changes = np.diff(active.astype(int), prepend=0, append=0)

    starts = np.where(changes == 1)[0]
    stops = np.where(changes == -1)[0]

    min_samples = int(min_duration_sec * sfreq)
    merge_gap_samples = int(merge_gap_sec * sfreq)

    intervals = []

    for start, stop in zip(starts, stops):
        if stop - start >= min_samples:
            intervals.append([start, stop])

    if not intervals:
        return []

    merged = [intervals[0]]

    for start, stop in intervals[1:]:
        previous_start, previous_stop = merged[-1]

        if start - previous_stop <= merge_gap_samples:
            merged[-1][1] = stop
        else:
            merged.append([start, stop])

    events = []

    flattened_data = data.transpose(1, 0, 2).reshape(data.shape[1], -1)

    for start, stop in merged:
        segment_global = z_signal[start:stop]
        segment_channels = flattened_data[:, start:stop]

        p2p_by_channel = np.ptp(segment_channels, axis=1)
        active_channels = p2p_by_channel > np.nanpercentile(p2p_by_channel, 75)

        events.append(
            {
                "start_sample": int(start),
                "stop_sample": int(stop),
                "duration_sec": float((stop - start) / sfreq),
                "z_peak": float(np.nanmax(segment_global)),
                "z_mean": float(np.nanmean(segment_global)),
                "peak_to_peak": float(np.nanmean(p2p_by_channel)),
                "line_length": compute_line_length(segment_global),
                "n_active_channels": int(np.sum(active_channels)),
            }
        )

    return events


def summarize_events(
    events: list[dict],
    total_duration_sec: float,
) -> dict:
    """
    Агрегирует события в признаки записи.

    Parameters
    ----------
    events : list[dict]
        Список событий.
    total_duration_sec : float
        Общая длительность записи.

    Returns
    -------
    dict
        Событийные признаки.
    """
    if not events or total_duration_sec <= 0:
        return {
            "n_events": 0,
            "event_rate_per_min": 0.0,
            "event_mean_duration_sec": np.nan,
            "event_median_duration_sec": np.nan,
            "event_mean_z_peak": np.nan,
            "event_mean_peak_to_peak": np.nan,
            "event_mean_line_length": np.nan,
            "event_mean_active_channels": np.nan,
        }

    durations = np.array([event["duration_sec"] for event in events])
    z_peaks = np.array([event["z_peak"] for event in events])
    p2p = np.array([event["peak_to_peak"] for event in events])
    line_length = np.array([event["line_length"] for event in events])
    active_channels = np.array([event["n_active_channels"] for event in events])

    return {
        "n_events": len(events),
        "event_rate_per_min": float(len(events) / total_duration_sec * 60),
        "event_mean_duration_sec": float(np.nanmean(durations)),
        "event_median_duration_sec": float(np.nanmedian(durations)),
        "event_mean_z_peak": float(np.nanmean(z_peaks)),
        "event_mean_peak_to_peak": float(np.nanmean(p2p)),
        "event_mean_line_length": float(np.nanmean(line_length)),
        "event_mean_active_channels": float(np.nanmean(active_channels)),
    }

In [ ]:
# @title 8.2. Массовый расчёт событийных признаков

event_features_path = OUT["tables"] / "event_features.csv"
event_features = load_cached_table(
    event_features_path,
    table_name="Event features",
)

if event_features is None:
    event_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расчёт событийных признаков",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])

            event_epochs = epochs.copy().filter(
                l_freq=CONFIG["event_band"][0],
                h_freq=CONFIG["event_band"][1],
                verbose=False,
            )

            data = event_epochs.get_data(copy=True)

            events = detect_events_from_global_signal(
                data=data,
                sfreq=sfreq,
                z_threshold=CONFIG["event_z_threshold"],
                min_duration_sec=CONFIG["event_min_duration_sec"],
                merge_gap_sec=CONFIG["event_merge_gap_sec"],
            )

            total_duration_sec = data.shape[0] * data.shape[2] / sfreq
            event_summary = summarize_events(
                events=events,
                total_duration_sec=total_duration_sec,
            )

            event_records.append(
                {
                    "group": row["group"],
                    "subject_id": row["subject_id"],
                    "record_id": row["record_id"],
                    **event_summary,
                }
            )

            del epochs
            del event_epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка событийного анализа для записи %s: %s",
                row.get("record_id"),
                error,
            )

    event_features = pd.DataFrame(event_records)
    event_features = save_computed_table(
        event_features,
        event_features_path,
        table_name="Event features",
    )

print("Event features shape:", event_features.shape)
display(event_features.head())


In [ ]:
# @title 8.3. Визуализация числа событий

if len(event_features) > 0:
    fig, ax = plt.subplots(figsize=(9, 6))

    data_to_plot = [
        event_features[event_features["group"] == group]["n_events"].dropna()
        for group in ["TBI", "Control"]
    ]

    box = ax.boxplot(
        data_to_plot,
        labels=[GROUP_LABELS_RU["TBI"], GROUP_LABELS_RU["Control"]],
        patch_artist=True,
        showfliers=False,
    )

    for patch, group_name in zip(box["boxes"], ["TBI", "Control"]):
        patch.set_facecolor(GROUP_COLORS[group_name])
        patch.set_alpha(1.0)

    ax.set_title("Число событий на запись")
    ax.set_ylabel("Количество событий")

    figure_path = OUT["figures"] / "event_n_events_boxplot.png"
    save_figure(fig, figure_path)

    plt.show()

# 9. Расширенный событийный анализ: event manifest и propagation_ms

In [ ]:
# @title 9.1. Детекция событий с ROI и propagation_ms

def get_channel_roi_map(roi_channels: dict[str, list[str]]) -> dict[str, str]:
    """
    Создаёт словарь channel -> ROI.

    Parameters
    ----------
    roi_channels : dict[str, list[str]]
        Словарь ROI.

    Returns
    -------
    dict[str, str]
        Соответствие каналов ROI.
    """
    mapping = {}

    for roi_name, channels in roi_channels.items():
        for channel in channels:
            mapping[channel] = roi_name

    return mapping


def detect_events_detailed(
    epochs: mne.Epochs,
    config: dict,
) -> list[dict]:
    """
    Детектирует события и рассчитывает детальные event-level признаки.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохи.
    config : dict
        Конфигурация.

    Returns
    -------
    list[dict]
        Список событий.
    """
    sfreq = float(epochs.info["sfreq"])

    event_epochs = epochs.copy().filter(
        l_freq=config["event_band"][0],
        h_freq=config["event_band"][1],
        verbose=False,
    )

    data = event_epochs.get_data(copy=True)
    n_epochs, n_channels, n_times = data.shape

    flat_data = data.transpose(1, 0, 2).reshape(n_channels, -1)
    global_signal = np.nanmean(np.abs(flat_data), axis=0)

    if np.nanstd(global_signal) == 0:
        return []

    z_signal = (
        global_signal - np.nanmean(global_signal)
    ) / np.nanstd(global_signal)

    active = z_signal >= config["event_z_threshold"]

    changes = np.diff(active.astype(int), prepend=0, append=0)
    starts = np.where(changes == 1)[0]
    stops = np.where(changes == -1)[0]

    min_samples = int(config["event_min_duration_sec"] * sfreq)
    merge_gap_samples = int(config["event_merge_gap_sec"] * sfreq)

    intervals = []

    for start, stop in zip(starts, stops):
        if stop - start >= min_samples:
            intervals.append([start, stop])

    if not intervals:
        return []

    merged = [intervals[0]]

    for start, stop in intervals[1:]:
        previous_start, previous_stop = merged[-1]

        if start - previous_stop <= merge_gap_samples:
            merged[-1][1] = stop
        else:
            merged.append([start, stop])

    channel_roi = get_channel_roi_map(config["roi_channels"])
    channel_names = epochs.ch_names

    events = []

    for event_index, (start, stop) in enumerate(merged):
        segment = flat_data[:, start:stop]
        segment_abs = np.abs(segment)

        p2p_by_channel = np.ptp(segment, axis=1)
        channel_threshold = np.nanpercentile(p2p_by_channel, 75)
        active_channels = p2p_by_channel >= channel_threshold

        active_channel_names = [
            channel
            for channel, is_active in zip(channel_names, active_channels)
            if is_active
        ]

        active_roi = sorted(
            {
                channel_roi[channel]
                for channel in active_channel_names
                if channel in channel_roi
            }
        )

        # Время достижения максимума по активным каналам.
        peak_latencies = []

        for channel_index, is_active in enumerate(active_channels):
            if not is_active:
                continue

            channel_segment = segment_abs[channel_index, :]
            peak_latencies.append(int(np.nanargmax(channel_segment)))

        if len(peak_latencies) >= 2:
            propagation_ms = (
                (np.max(peak_latencies) - np.min(peak_latencies))
                / sfreq
                * 1000
            )
        else:
            propagation_ms = np.nan

        event_global = z_signal[start:stop]

        events.append(
            {
                "event_id": event_index,
                "start_sample": int(start),
                "stop_sample": int(stop),
                "duration_sec": float((stop - start) / sfreq),
                "z_peak": float(np.nanmax(event_global)),
                "z_mean": float(np.nanmean(event_global)),
                "peak_to_peak": float(np.nanmean(p2p_by_channel)),
                "line_length": compute_line_length(event_global),
                "area": float(np.trapz(np.abs(event_global), dx=1 / sfreq)),
                "n_active_channels": int(np.sum(active_channels)),
                "max_active_channels": int(np.sum(active_channels)),
                "n_active_roi": int(len(active_roi)),
                "active_roi": "|".join(active_roi),
                "spatial_spread_ratio": float(
                    np.sum(active_channels) / len(active_channels)
                ),
                "propagation_ms": float(propagation_ms),
            }
        )

    return events

In [ ]:
# @title 9.2. Event manifest и расширенные событийные признаки

event_manifest_path = OUT["tables"] / "event_manifest_long.csv"
event_features_extended_path = OUT["tables"] / "event_features_extended.csv"

event_manifest_long = load_cached_table(
    event_manifest_path,
    table_name="Event manifest",
)
event_features_extended = load_cached_table(
    event_features_extended_path,
    table_name="Extended event features",
)

if event_manifest_long is None or event_features_extended is None:
    event_manifest_records = []
    event_summary_records = []

    for _, row in tqdm(
        analysis_inventory.iterrows(),
        total=len(analysis_inventory),
        desc="Расширенный событийный анализ",
    ):
        try:
            epochs = load_epochs_file(row["common_epochs_path"])
            epochs = select_epochs_subset(
                epochs=epochs,
                max_epochs=CONFIG["max_epochs_per_record"],
                random_state=CONFIG["epoch_random_state"],
            )

            sfreq = float(epochs.info["sfreq"])
            total_duration_sec = len(epochs) * len(epochs.times) / sfreq

            events = detect_events_detailed(
                epochs=epochs,
                config=CONFIG,
            )

            for event in events:
                event_manifest_records.append(
                    {
                        "group": row["group"],
                        "subject_id": row["subject_id"],
                        "record_id": row["record_id"],
                        **event,
                    }
                )

            if len(events) == 0:
                summary = {
                    "n_events": 0,
                    "event_rate_per_min": 0.0,
                    "event_mean_duration_sec": np.nan,
                    "event_mean_z_peak": np.nan,
                    "event_mean_peak_to_peak": np.nan,
                    "event_mean_line_length": np.nan,
                    "event_mean_area": np.nan,
                    "event_mean_active_channels": np.nan,
                    "event_mean_active_roi": np.nan,
                    "event_mean_spatial_spread_ratio": np.nan,
                    "event_mean_propagation_ms": np.nan,
                }
            else:
                event_df = pd.DataFrame(events)

                summary = {
                    "n_events": len(events),
                    "event_rate_per_min": len(events) / total_duration_sec * 60,
                    "event_mean_duration_sec": event_df["duration_sec"].mean(),
                    "event_mean_z_peak": event_df["z_peak"].mean(),
                    "event_mean_peak_to_peak": event_df["peak_to_peak"].mean(),
                    "event_mean_line_length": event_df["line_length"].mean(),
                    "event_mean_area": event_df["area"].mean(),
                    "event_mean_active_channels": (
                        event_df["n_active_channels"].mean()
                    ),
                    "event_mean_active_roi": event_df["n_active_roi"].mean(),
                    "event_mean_spatial_spread_ratio": (
                        event_df["spatial_spread_ratio"].mean()
                    ),
                    "event_mean_propagation_ms": (
                        event_df["propagation_ms"].mean()
                    ),
                }

            event_summary_records.append(
                {
                    "group": row["group"],
                    "subject_id": row["subject_id"],
                    "record_id": row["record_id"],
                    **summary,
                }
            )

            del epochs
            gc.collect()

        except Exception as error:
            logger.exception(
                "Ошибка расширенного событийного анализа для записи %s: %s",
                row.get("record_id"),
                error,
            )

    event_manifest_long = pd.DataFrame(event_manifest_records)
    event_features_extended = pd.DataFrame(event_summary_records)

    event_manifest_long = save_computed_table(
        event_manifest_long,
        event_manifest_path,
        table_name="Event manifest",
    )

    event_features_extended = save_computed_table(
        event_features_extended,
        event_features_extended_path,
        table_name="Extended event features",
    )

print("Event manifest shape:", event_manifest_long.shape)
print("Extended event features shape:", event_features_extended.shape)
display(event_features_extended.head())


In [ ]:
# @title 9.3. Событийные признаки: морфология и пространственное распространение

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.stats.multitest import multipletests


# ---------------------------------------------------------------------
# Выбор таблицы событийных признаков
# ---------------------------------------------------------------------
if "event_features_extended" in globals():
    event_plot_df = event_features_extended.copy()
    event_source_name = "event_features_extended"
elif "event_features" in globals():
    event_plot_df = event_features.copy()
    event_source_name = "event_features"
else:
    raise NameError(
        "Не найдена таблица событийных признаков. "
        "Ожидается `event_features_extended` или `event_features`."
    )

print(f"Используется таблица: {event_source_name}")


# ---------------------------------------------------------------------
# Цвета и подписи
# ---------------------------------------------------------------------
COL_TBI = "#A6192E"
COL_CONTROL = "#1F497D"

group_order = ["Control", "TBI"]

group_labels_ru = {
    "Control": "Контроль",
    "TBI": "ЧМТ",
}

group_colors = {
    "Control": COL_CONTROL,
    "TBI": COL_TBI,
}


# ---------------------------------------------------------------------
# Признаки для рисунка
# ---------------------------------------------------------------------
features_to_plot = [
    "event_mean_duration_sec",
    "event_mean_peak_to_peak",
    "event_mean_line_length",
    "event_mean_z_peak",
    "event_mean_active_roi",
    "event_mean_propagation_ms",
]

feature_labels = {
    "event_mean_duration_sec": "Длительность события",
    "event_mean_peak_to_peak": "Peak-to-peak амплитуда",
    "event_mean_line_length": "Line length",
    "event_mean_z_peak": "Пиковая амплитуда, z",
    "event_mean_active_roi": "Число вовлечённых ROI",
    "event_mean_spatial_spread_ratio": "Доля вовлечённых ROI",
    "event_mean_propagation_ms": "Время распространения",
}

feature_ylabels = {
    "event_mean_duration_sec": "с",
    "event_mean_peak_to_peak": "мкВ",
    "event_mean_line_length": "у.е.",
    "event_mean_z_peak": "z",
    "event_mean_active_roi": "ROI",
    "event_mean_spatial_spread_ratio": "доля",
    "event_mean_propagation_ms": "мс",
}

available_features = [
    feature for feature in features_to_plot
    if feature in event_plot_df.columns
]

if not available_features:
    raise ValueError(
        "Не найден ни один ожидаемый событийный признак. "
        f"Доступные колонки:\n{list(event_plot_df.columns)}"
    )


# ---------------------------------------------------------------------
# Проверка обязательных колонок
# ---------------------------------------------------------------------
required_columns = ["group", "subject_id"]

missing_columns = [
    column for column in required_columns
    if column not in event_plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В таблице событийных признаков отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )


# ---------------------------------------------------------------------
# Нормализация групп
# ---------------------------------------------------------------------
event_plot_df["group_norm"] = event_plot_df["group"].map(
    {
        "TBI": "TBI",
        "tbi": "TBI",
        "ЧМТ": "TBI",
        "Control": "Control",
        "control": "Control",
        "Контроль": "Control",
    }
)

event_plot_df = event_plot_df[
    event_plot_df["group_norm"].isin(group_order)
].copy()


# ---------------------------------------------------------------------
# Subject-level агрегация
# ---------------------------------------------------------------------
# Здесь используется медиана по событиям/записям внутри субъекта.
# Это устойчивее к редким очень крупным событиям.
# ---------------------------------------------------------------------
event_subject = (
    event_plot_df
    .replace([np.inf, -np.inf], np.nan)
    .groupby(["group_norm", "subject_id"], as_index=False)
    .agg(
        {
            feature: "median"
            for feature in available_features
        }
    )
)

print("Event-level / record-level:", event_plot_df.shape)
print("Subject-level:", event_subject.shape)

display(
    event_subject
    .groupby("group_norm")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)


# ---------------------------------------------------------------------
# Статистические функции
# ---------------------------------------------------------------------
def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    """Рассчитывает Hedges' g для двух независимых групп."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_sd = np.sqrt(
        (
            (n_x - 1) * np.var(x, ddof=1)
            + (n_y - 1) * np.var(y, ddof=1)
        )
        / (n_x + n_y - 2)
    )

    if pooled_sd == 0:
        return np.nan

    cohen_d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (n_x + n_y) - 9)

    return float(cohen_d * correction)


# ---------------------------------------------------------------------
# Статистика по признакам
# ---------------------------------------------------------------------
stats_records = []

for feature in available_features:
    control_values = (
        event_subject.loc[event_subject["group_norm"] == "Control", feature]
        .dropna()
        .to_numpy(dtype=float)
    )

    tbi_values = (
        event_subject.loc[event_subject["group_norm"] == "TBI", feature]
        .dropna()
        .to_numpy(dtype=float)
    )

    if len(control_values) < 2 or len(tbi_values) < 2:
        p_value = np.nan
        effect = np.nan
    else:
        _, p_value = stats.mannwhitneyu(
            tbi_values,
            control_values,
            alternative="two-sided",
        )
        effect = hedges_g(tbi_values, control_values)

    stats_records.append(
        {
            "feature": feature,
            "p_value": p_value,
            "hedges_g": effect,
            "n_tbi": len(tbi_values),
            "n_control": len(control_values),
        }
    )

event_morphology_stats = pd.DataFrame(stats_records)

event_morphology_stats["q_fdr"] = multipletests(
    event_morphology_stats["p_value"].fillna(1.0),
    method="fdr_bh",
)[1]

display(event_morphology_stats)


# ---------------------------------------------------------------------
# Вспомогательные функции для графика
# ---------------------------------------------------------------------
def q_to_stars(q_value: float) -> str:
    """Преобразует q-FDR в звёздочки."""
    if pd.isna(q_value):
        return ""

    if q_value < 0.001:
        return "***"

    if q_value < 0.01:
        return "**"

    if q_value < 0.05:
        return "*"

    return ""


def draw_boxplot(ax, data, positions, color, width=0.42):
    """Рисует boxplot в едином стиле."""
    ax.boxplot(
        data,
        positions=positions,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": color,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": color,
            "markeredgecolor": color,
            "markersize": 2.8,
            "alpha": 0.55,
        },
    )


# ---------------------------------------------------------------------
# Фигура 2×3
# ---------------------------------------------------------------------
n_cols = 3
n_rows = int(np.ceil(len(available_features) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(13.5, 7.8),
)

axes = np.asarray(axes).reshape(n_rows, n_cols)

fig.suptitle(
    "Морфологические и пространственные признаки событий",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

for idx, feature in enumerate(available_features):
    ax = axes[idx // n_cols, idx % n_cols]

    control_values = (
        event_subject.loc[event_subject["group_norm"] == "Control", feature]
        .dropna()
        .to_numpy(dtype=float)
    )

    tbi_values = (
        event_subject.loc[event_subject["group_norm"] == "TBI", feature]
        .dropna()
        .to_numpy(dtype=float)
    )

    draw_boxplot(
        ax=ax,
        data=[control_values],
        positions=[0],
        color=COL_CONTROL,
    )

    draw_boxplot(
        ax=ax,
        data=[tbi_values],
        positions=[1],
        color=COL_TBI,
    )

    stat_row = event_morphology_stats[
        event_morphology_stats["feature"] == feature
    ].iloc[0]

    stars = q_to_stars(stat_row["q_fdr"])

    ax.set_title(
        (
            f"{feature_labels.get(feature, feature)}\n"
            f"{stars} q={stat_row['q_fdr']:.3g}, "
            f"g={stat_row['hedges_g']:.2f}"
        ),
        fontsize=10.5,
        fontweight="bold",
        pad=6,
    )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Контроль", "ЧМТ"])

    ax.set_ylabel(feature_ylabels.get(feature, ""))

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.08)

    # Robust-ось для признаков с тяжёлыми выбросами
    values = np.concatenate([control_values, tbi_values])

    values = values[np.isfinite(values)]

    if len(values) > 0:
        y_low = np.nanpercentile(values, 0.5)
        y_high = np.nanpercentile(values, 99.2)
        y_range = y_high - y_low

        if np.isfinite(y_range) and y_range > 0:
            lower = y_low - 0.05 * y_range
            upper = y_high + 0.12 * y_range

            if y_low >= 0:
                lower = max(0, lower)

            ax.set_ylim(lower, upper)

# Скрываем пустые панели, если признаков меньше 6
for empty_idx in range(len(available_features), n_rows * n_cols):
    axes[empty_idx // n_cols, empty_idx % n_cols].axis("off")


# ---------------------------------------------------------------------
# Общая легенда
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.subplots_adjust(
    left=0.07,
    right=0.98,
    top=0.88,
    bottom=0.12,
    wspace=0.28,
    hspace=0.45,
)


# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "event_morphology_spatial_features_subject_level.png"

save_figure(fig, figure_path)

plt.show()

# 10. Финальная сборка таблицы признаков

Все расчётные блоки выше формируют отдельные long-таблицы. Итоговая record-level таблица собирается один раз в конце расчётной части, чтобы в неё вошли DFA, burst, connectivity, graph metrics и событийные признаки.

In [ ]:
# @title 10.1. Функции преобразования признаков в wide-формат

def long_to_wide(
    df: pd.DataFrame,
    id_columns: list[str],
    context_columns: list[str],
    feature_columns: list[str],
    block_name: str,
) -> pd.DataFrame:
    """
    Преобразует long-таблицу признаков в wide-формат.
    """
    records = []

    for _, row in df.iterrows():
        base = {column: row[column] for column in id_columns}

        context = "_".join(
            str(row[column])
            for column in context_columns
        )

        for feature in feature_columns:
            records.append(
                {
                    **base,
                    "wide_feature": f"{block_name}__{context}__{feature}",
                    "value": row[feature],
                }
            )

    long_features = pd.DataFrame(records)

    if long_features.empty:
        return pd.DataFrame(columns=id_columns)

    wide = (
        long_features
        .pivot_table(
            index=id_columns,
            columns="wide_feature",
            values="value",
            aggfunc="mean",
        )
        .reset_index()
    )

    wide.columns.name = None

    return wide


def get_numeric_feature_columns(
    df: pd.DataFrame,
    exclude_columns: list[str],
) -> list[str]:
    """
    Возвращает числовые признаки, исключая служебные колонки.
    """
    return [
        column for column in df.columns
        if column not in exclude_columns
        and pd.api.types.is_numeric_dtype(df[column])
    ]


print("Функции преобразования признаков в wide-формат загружены.")


In [ ]:
# @title 10.2. Финальное формирование итоговой record-level таблицы признаков

id_columns = ["group", "subject_id", "record_id"]
final_features_path = OUT["tables"] / "temporal_connectivity_event_features_record_level.csv"

# Если итоговая таблица уже есть и кэширование включено, считываем её.
temporal_connectivity_event_features = load_cached_table(
    final_features_path,
    table_name="Final temporal/connectivity/event feature table",
)

if temporal_connectivity_event_features is None:
    wide_tables = []

    # -----------------------------------------------------------------
    # 1. DFA / LRTC
    # -----------------------------------------------------------------
    if "dfa_features_long" in globals():
        dfa_wide = long_to_wide(
            df=dfa_features_long,
            id_columns=id_columns,
            context_columns=["roi", "band"],
            feature_columns=["dfa_exponent"],
            block_name="dfa",
        )
        wide_tables.append(dfa_wide)
        print(f"DFA wide: {dfa_wide.shape}")
    else:
        print("DFA-признаки не найдены: dfa_features_long отсутствует.")

    # -----------------------------------------------------------------
    # 2. Burst-признаки
    # -----------------------------------------------------------------
    if "burst_features_long" in globals():
        burst_feature_columns = get_numeric_feature_columns(
            df=burst_features_long,
            exclude_columns=id_columns + ["roi", "band"],
        )

        burst_wide = long_to_wide(
            df=burst_features_long,
            id_columns=id_columns,
            context_columns=["roi", "band"],
            feature_columns=burst_feature_columns,
            block_name="burst",
        )
        wide_tables.append(burst_wide)
        print(f"Burst wide: {burst_wide.shape}")
    else:
        print("Burst-признаки не найдены: burst_features_long отсутствует.")

    # -----------------------------------------------------------------
    # 3. Функциональная связность
    # -----------------------------------------------------------------
    if "connectivity_features_long" in globals():
        conn_feature_columns = get_numeric_feature_columns(
            df=connectivity_features_long,
            exclude_columns=id_columns + ["band", "method"],
        )

        connectivity_wide = long_to_wide(
            df=connectivity_features_long,
            id_columns=id_columns,
            context_columns=["band", "method"],
            feature_columns=conn_feature_columns,
            block_name="conn",
        )
        wide_tables.append(connectivity_wide)
        print(f"Connectivity wide: {connectivity_wide.shape}")
    else:
        print("Connectivity-признаки не найдены: connectivity_features_long отсутствует.")

    # -----------------------------------------------------------------
    # 4. Базовые событийные признаки
    # -----------------------------------------------------------------
    if "event_features" in globals():
        event_feature_columns = get_numeric_feature_columns(
            df=event_features,
            exclude_columns=id_columns,
        )

        event_wide = event_features[id_columns + event_feature_columns].copy()
        event_wide = event_wide.rename(
            columns={
                column: f"event__{column}"
                for column in event_feature_columns
            }
        )
        wide_tables.append(event_wide)
        print(f"Event wide: {event_wide.shape}")
    else:
        print("Базовые event-признаки не найдены: event_features отсутствует.")

    # -----------------------------------------------------------------
    # 5. Расширенные graph metrics
    # -----------------------------------------------------------------
    if "extended_graph_features_long" in globals():
        extended_graph_numeric = get_numeric_feature_columns(
            df=extended_graph_features_long,
            exclude_columns=id_columns + ["band", "method"],
        )

        extended_graph_wide = long_to_wide(
            df=extended_graph_features_long,
            id_columns=id_columns,
            context_columns=["band", "method"],
            feature_columns=extended_graph_numeric,
            block_name="graph_ext",
        )
        wide_tables.append(extended_graph_wide)
        print(f"Extended graph wide: {extended_graph_wide.shape}")
    else:
        print("Расширенные graph metrics не найдены: extended_graph_features_long отсутствует.")

    # -----------------------------------------------------------------
    # 6. Расширенные событийные признаки
    # -----------------------------------------------------------------
    if "event_features_extended" in globals():
        event_extended_numeric = get_numeric_feature_columns(
            df=event_features_extended,
            exclude_columns=id_columns,
        )

        event_extended_wide = event_features_extended[
            id_columns + event_extended_numeric
        ].copy()

        event_extended_wide = event_extended_wide.rename(
            columns={
                column: f"event_ext__{column}"
                for column in event_extended_numeric
            }
        )
        wide_tables.append(event_extended_wide)
        print(f"Extended event wide: {event_extended_wide.shape}")
    else:
        print("Расширенные event-признаки не найдены: event_features_extended отсутствует.")

    if not wide_tables:
        raise ValueError("Не найдено ни одной таблицы признаков для объединения.")

    temporal_connectivity_event_features = wide_tables[0].copy()

    for table in wide_tables[1:]:
        temporal_connectivity_event_features = (
            temporal_connectivity_event_features
            .merge(table, on=id_columns, how="outer")
        )

    temporal_connectivity_event_features = save_computed_table(
        temporal_connectivity_event_features,
        final_features_path,
        table_name="Final temporal/connectivity/event feature table",
    )

# Алиас для совместимости со статистическими ячейками.
temporal_connectivity_event_features_v2 = temporal_connectivity_event_features.copy()

print(
    "Итоговая record-level таблица признаков:",
    temporal_connectivity_event_features.shape,
)
display(temporal_connectivity_event_features.head())


In [ ]:
# @title 10.3. Карта признаков и анализ пропусков

feature_map_path = OUT["tables"] / "temporal_connectivity_event_feature_map.csv"
missing_summary_path = OUT["tables"] / "temporal_connectivity_event_feature_missingness.csv"

feature_map = load_cached_table(
    feature_map_path,
    table_name="Feature map",
)
missing_summary = load_cached_table(
    missing_summary_path,
    table_name="Feature missingness summary",
)

if feature_map is None or missing_summary is None:
    feature_columns = [
        column for column in temporal_connectivity_event_features.columns
        if column not in ["group", "subject_id", "record_id"]
    ]

    feature_map = pd.DataFrame(
        {
            "feature": feature_columns,
            "block": [
                feature.split("__")[0]
                if "__" in feature else "unknown"
                for feature in feature_columns
            ],
        }
    )

    missing_summary = pd.DataFrame(
        {
            "feature": feature_columns,
            "missing_prop": temporal_connectivity_event_features[
                feature_columns
            ].isna().mean().values,
            "n_missing": temporal_connectivity_event_features[
                feature_columns
            ].isna().sum().values,
            "n_non_missing": temporal_connectivity_event_features[
                feature_columns
            ].notna().sum().values,
            "zero_variance": [
                temporal_connectivity_event_features[column].nunique(
                    dropna=True
                ) <= 1
                for column in feature_columns
            ],
        }
    )

    feature_map = save_computed_table(
        feature_map,
        feature_map_path,
        table_name="Feature map",
    )

    missing_summary = save_computed_table(
        missing_summary,
        missing_summary_path,
        table_name="Feature missingness summary",
    )

print("Число итоговых признаков:", len(feature_map))
print("Размер feature map:", feature_map.shape)
print("Размер missingness summary:", missing_summary.shape)

display(feature_map.head(20))
display(
    missing_summary
    .sort_values("missing_prop", ascending=False)
    .head(30)
)


# 11. Статистический анализ

Основная статистическая интерпретация выполняется на subject-level: если у субъекта несколько записей, признаки предварительно усредняются внутри субъекта.

In [ ]:
# @title 11.1. Статистические функции

def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    """
    Рассчитывает Hedges' g для двух независимых групп.

    Parameters
    ----------
    x : np.ndarray
        Значения первой группы.
    y : np.ndarray
        Значения второй группы.

    Returns
    -------
    float
        Hedges' g.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_sd = np.sqrt(
        (
            (n_x - 1) * np.var(x, ddof=1)
            + (n_y - 1) * np.var(y, ddof=1)
        )
        / (n_x + n_y - 2)
    )

    if pooled_sd == 0:
        return np.nan

    cohen_d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (n_x + n_y) - 9)

    return float(cohen_d * correction)


def rank_biserial_from_u(
    u_stat: float,
    n_x: int,
    n_y: int,
) -> float:
    """
    Рассчитывает rank-biserial correlation из Mann–Whitney U.

    Значение > 0 означает, что значения первой группы в среднем выше,
    чем значения второй группы.

    Parameters
    ----------
    u_stat : float
        Статистика U для первой группы.
    n_x : int
        Размер первой группы.
    n_y : int
        Размер второй группы.

    Returns
    -------
    float
        Rank-biserial correlation.
    """
    if n_x == 0 or n_y == 0:
        return np.nan

    return float((2 * u_stat) / (n_x * n_y) - 1)


def normalize_stat_group(value: str) -> str:
    """
    Приводит названия групп к TBI / Control.
    """
    mapping = {
        "TBI": "TBI",
        "tbi": "TBI",
        "ЧМТ": "TBI",
        "patient": "TBI",
        "patients": "TBI",
        "Control": "Control",
        "control": "Control",
        "Контроль": "Control",
        "CTRL": "Control",
        "ctl": "Control",
        "HC": "Control",
        "healthy": "Control",
    }

    return mapping.get(str(value).strip(), str(value).strip())


def safe_nanmean(values: np.ndarray) -> float:
    """Безопасно считает среднее."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.nanmean(values))


def safe_nanmedian(values: np.ndarray) -> float:
    """Безопасно считает медиану."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.nanmedian(values))


def safe_iqr(values: np.ndarray) -> float:
    """Безопасно считает IQR."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(
        np.nanpercentile(values, 75)
        - np.nanpercentile(values, 25)
    )


def compare_groups_by_features(
    df: pd.DataFrame,
    feature_columns: list[str],
    context_columns: list[str] | None = None,
) -> pd.DataFrame:
    """
    Сравнивает группы TBI и Control по набору признаков.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков. Должна содержать колонку group.
    feature_columns : list[str]
        Список числовых признаков.
    context_columns : list[str] | None
        Контекстные колонки для группировки.

    Returns
    -------
    pd.DataFrame
        Таблица статистических сравнений.
    """
    context_columns = context_columns or []

    if "group" not in df.columns:
        raise ValueError("В таблице df отсутствует обязательная колонка 'group'.")

    missing_features = [
        feature for feature in feature_columns
        if feature not in df.columns
    ]

    if missing_features:
        raise ValueError(
            "В таблице df отсутствуют признаки: "
            f"{missing_features}"
        )

    work_df = df.copy()
    work_df["group"] = work_df["group"].map(normalize_stat_group)

    records = []

    if context_columns:
        iterator = work_df.groupby(context_columns, dropna=False)
    else:
        iterator = [((), work_df)]

    for context_key, sub_df in iterator:
        if context_columns and not isinstance(context_key, tuple):
            context_key = (context_key,)

        if context_columns:
            context = dict(zip(context_columns, context_key))
        else:
            context = {}

        for feature in feature_columns:
            tbi_values = (
                sub_df.loc[sub_df["group"] == "TBI", feature]
                .to_numpy(dtype=float)
            )
            ctl_values = (
                sub_df.loc[sub_df["group"] == "Control", feature]
                .to_numpy(dtype=float)
            )

            tbi_values = tbi_values[np.isfinite(tbi_values)]
            ctl_values = ctl_values[np.isfinite(ctl_values)]

            if len(tbi_values) < 2 or len(ctl_values) < 2:
                u_stat = np.nan
                p_value = np.nan
                effect = np.nan
                rank_biserial = np.nan
            else:
                u_stat, p_value = stats.mannwhitneyu(
                    tbi_values,
                    ctl_values,
                    alternative="two-sided",
                )
                effect = hedges_g(tbi_values, ctl_values)
                rank_biserial = rank_biserial_from_u(
                    u_stat=u_stat,
                    n_x=len(tbi_values),
                    n_y=len(ctl_values),
                )

            tbi_mean = safe_nanmean(tbi_values)
            control_mean = safe_nanmean(ctl_values)
            tbi_median = safe_nanmedian(tbi_values)
            control_median = safe_nanmedian(ctl_values)

            records.append(
                {
                    **context,
                    "feature": feature,
                    "tbi_mean": tbi_mean,
                    "control_mean": control_mean,
                    "delta_tbi_minus_control": (
                        tbi_mean - control_mean
                    ),
                    "tbi_median": tbi_median,
                    "control_median": control_median,
                    "delta_median_tbi_minus_control": (
                        tbi_median - control_median
                    ),
                    "tbi_iqr": safe_iqr(tbi_values),
                    "control_iqr": safe_iqr(ctl_values),
                    "hedges_g": effect,
                    "rank_biserial": rank_biserial,
                    "u_stat": u_stat,
                    "p_value": p_value,
                    "n_tbi": len(tbi_values),
                    "n_control": len(ctl_values),
                }
            )

    result = pd.DataFrame(records)

    if len(result) > 0:
        result["q_fdr"] = multipletests(
            result["p_value"].fillna(1.0),
            method="fdr_bh",
        )[1]

        result["significant_fdr_0_05"] = result["q_fdr"] < 0.05

    return result


print("Статистические функции загружены.")

In [ ]:
# @title 11.2. Subject-level агрегация временных, сетевых и событийных признаков

# ---------------------------------------------------------------------
# Выбираем наиболее полную итоговую таблицу признаков
# ---------------------------------------------------------------------
if "temporal_connectivity_event_features_v2" in globals():
    features_record = temporal_connectivity_event_features_v2.copy()
    source_table_name = "temporal_connectivity_event_features_v2"
elif "temporal_connectivity_event_features" in globals():
    features_record = temporal_connectivity_event_features.copy()
    source_table_name = "temporal_connectivity_event_features"
else:
    raise NameError(
        "Не найдена итоговая таблица признаков. Ожидается "
        "`temporal_connectivity_event_features_v2` или "
        "`temporal_connectivity_event_features`."
    )

print(f"Используется таблица: {source_table_name}")
print("Record-level shape:", features_record.shape)

# ---------------------------------------------------------------------
# Проверяем обязательные колонки
# ---------------------------------------------------------------------
required_columns = ["group", "subject_id"]

missing_columns = [
    col for col in required_columns
    if col not in features_record.columns
]

if missing_columns:
    raise ValueError(
        "В итоговой таблице отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

# ---------------------------------------------------------------------
# Числовые признаки
# ---------------------------------------------------------------------
id_columns = [
    col for col in ["group", "subject_id", "record_id"]
    if col in features_record.columns
]

feature_columns_all = [
    col for col in features_record.columns
    if col not in id_columns
    and pd.api.types.is_numeric_dtype(features_record[col])
]

print(f"Число числовых признаков: {len(feature_columns_all)}")

# ---------------------------------------------------------------------
# Subject-level агрегация
# ---------------------------------------------------------------------
features_subject = (
    features_record
    .groupby(["group", "subject_id"], as_index=False)
    .mean(numeric_only=True)
)

print("Subject-level shape:", features_subject.shape)

display(
    features_subject
    .groupby("group")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)

save_table(
    features_subject,
    OUT["tables"] / "temporal_connectivity_event_features_subject_level.csv",
)

display(features_subject.head())

In [ ]:
# @title 11.3. Subject-level статистика по всем временным, сетевым и событийным признакам

temporal_event_stats_subject = compare_groups_by_features(
    df=features_subject,
    feature_columns=feature_columns_all,
    context_columns=None,
)

temporal_event_stats_subject = temporal_event_stats_subject.sort_values(
    ["q_fdr", "feature"]
)

save_table(
    temporal_event_stats_subject,
    OUT["tables"] / "temporal_connectivity_event_stats_subject_level.csv",
)

print("Число проверенных признаков:", len(temporal_event_stats_subject))
print(
    "Значимых после FDR q < 0.05:",
    int(temporal_event_stats_subject["significant_fdr_0_05"].sum()),
)

display(temporal_event_stats_subject.head(30))

In [ ]:
# @title 11.4. Subject-level статистика DFA по ROI и диапазонам

if "dfa_features_long" not in globals():
    raise NameError("Не найдена таблица dfa_features_long.")

dfa_stat_df = dfa_features_long.copy()

required_columns = [
    "group",
    "subject_id",
    "roi",
    "band",
    "dfa_exponent",
]

missing_columns = [
    col for col in required_columns
    if col not in dfa_stat_df.columns
]

if missing_columns:
    raise ValueError(
        "В dfa_features_long отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

dfa_subject = (
    dfa_stat_df
    .groupby(["group", "subject_id", "roi", "band"], as_index=False)
    .agg(
        dfa_exponent=("dfa_exponent", "mean"),
    )
)

dfa_stats_subject = compare_groups_by_features(
    df=dfa_subject,
    feature_columns=["dfa_exponent"],
    context_columns=["roi", "band"],
)

dfa_stats_subject = dfa_stats_subject.sort_values(
    ["q_fdr", "roi", "band"]
)

save_table(
    dfa_subject,
    OUT["tables"] / "dfa_features_subject_level.csv",
)

save_table(
    dfa_stats_subject,
    OUT["tables"] / "dfa_stats_subject_level.csv",
)

print("DFA subject-level:")
display(
    dfa_subject
    .groupby("group")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)

display(dfa_stats_subject)

In [ ]:
# @title 11.5. Subject-level статистика burst-признаков по ROI и диапазонам

if "burst_features_long" not in globals():
    raise NameError("Не найдена таблица burst_features_long.")

burst_stat_df = burst_features_long.copy()

required_columns = [
    "group",
    "subject_id",
    "roi",
    "band",
]

missing_columns = [
    col for col in required_columns
    if col not in burst_stat_df.columns
]

if missing_columns:
    raise ValueError(
        "В burst_features_long отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

burst_numeric_columns = [
    col for col in burst_stat_df.columns
    if col not in ["group", "subject_id", "record_id", "roi", "band"]
    and pd.api.types.is_numeric_dtype(burst_stat_df[col])
]

burst_subject = (
    burst_stat_df
    .groupby(["group", "subject_id", "roi", "band"], as_index=False)
    .mean(numeric_only=True)
)

burst_stats_subject = compare_groups_by_features(
    df=burst_subject,
    feature_columns=burst_numeric_columns,
    context_columns=["roi", "band"],
)

burst_stats_subject = burst_stats_subject.sort_values(
    ["q_fdr", "roi", "band", "feature"]
)

save_table(
    burst_subject,
    OUT["tables"] / "burst_features_subject_level.csv",
)

save_table(
    burst_stats_subject,
    OUT["tables"] / "burst_stats_subject_level.csv",
)

print("Burst subject-level:")
display(
    burst_subject
    .groupby("group")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)

display(burst_stats_subject.head(30))

In [ ]:
# @title 11.6. Subject-level статистика connectivity-признаков по методу и диапазону

if "connectivity_features_long" not in globals():
    raise NameError("Не найдена таблица connectivity_features_long.")

conn_stat_df = connectivity_features_long.copy()

required_columns = [
    "group",
    "subject_id",
    "method",
    "band",
]

missing_columns = [
    col for col in required_columns
    if col not in conn_stat_df.columns
]

if missing_columns:
    raise ValueError(
        "В connectivity_features_long отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

conn_numeric_columns = [
    col for col in conn_stat_df.columns
    if col not in ["group", "subject_id", "record_id", "method", "band"]
    and pd.api.types.is_numeric_dtype(conn_stat_df[col])
]

connectivity_subject = (
    conn_stat_df
    .groupby(["group", "subject_id", "method", "band"], as_index=False)
    .mean(numeric_only=True)
)

connectivity_stats_subject = compare_groups_by_features(
    df=connectivity_subject,
    feature_columns=conn_numeric_columns,
    context_columns=["method", "band"],
)

connectivity_stats_subject = connectivity_stats_subject.sort_values(
    ["q_fdr", "method", "band", "feature"]
)

save_table(
    connectivity_subject,
    OUT["tables"] / "connectivity_features_subject_level.csv",
)

save_table(
    connectivity_stats_subject,
    OUT["tables"] / "connectivity_stats_subject_level.csv",
)

print("Connectivity subject-level:")
display(
    connectivity_subject
    .groupby("group")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)

display(connectivity_stats_subject.head(30))

In [ ]:
# @title 11.7. Subject-level статистика событийных признаков

# ---------------------------------------------------------------------
# Используем расширенные события, если они уже рассчитаны
# ---------------------------------------------------------------------
if "event_features_extended" in globals():
    event_stat_df = event_features_extended.copy()
    event_source_name = "event_features_extended"
elif "event_features" in globals():
    event_stat_df = event_features.copy()
    event_source_name = "event_features"
else:
    raise NameError(
        "Не найдены событийные признаки. Ожидается "
        "`event_features_extended` или `event_features`."
    )

print(f"Используется таблица: {event_source_name}")

required_columns = [
    "group",
    "subject_id",
]

missing_columns = [
    col for col in required_columns
    if col not in event_stat_df.columns
]

if missing_columns:
    raise ValueError(
        "В таблице событийных признаков отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

event_numeric_columns = [
    col for col in event_stat_df.columns
    if col not in ["group", "subject_id", "record_id"]
    and pd.api.types.is_numeric_dtype(event_stat_df[col])
]

event_subject = (
    event_stat_df
    .groupby(["group", "subject_id"], as_index=False)
    .mean(numeric_only=True)
)

event_stats_subject = compare_groups_by_features(
    df=event_subject,
    feature_columns=event_numeric_columns,
    context_columns=None,
)

event_stats_subject = event_stats_subject.sort_values(
    ["q_fdr", "feature"]
)

save_table(
    event_subject,
    OUT["tables"] / "event_features_subject_level.csv",
)

save_table(
    event_stats_subject,
    OUT["tables"] / "event_stats_subject_level.csv",
)

print("Event subject-level:")
display(
    event_subject
    .groupby("group")["subject_id"]
    .nunique()
    .to_frame("n_subjects")
)

display(event_stats_subject.head(30))

In [ ]:
# @title 11.8. Subject-level статистика расширенных graph metrics

if "extended_graph_features_long" not in globals():
    print("Таблица extended_graph_features_long не найдена. Ячейка пропущена.")
else:
    graph_stat_df = extended_graph_features_long.copy()

    required_columns = [
        "group",
        "subject_id",
        "method",
        "band",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in graph_stat_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "В extended_graph_features_long отсутствуют обязательные колонки: "
            f"{missing_columns}"
        )

    graph_numeric_columns = [
        col for col in graph_stat_df.columns
        if col not in ["group", "subject_id", "record_id", "method", "band"]
        and pd.api.types.is_numeric_dtype(graph_stat_df[col])
    ]

    graph_subject = (
        graph_stat_df
        .groupby(["group", "subject_id", "method", "band"], as_index=False)
        .mean(numeric_only=True)
    )

    graph_stats_subject = compare_groups_by_features(
        df=graph_subject,
        feature_columns=graph_numeric_columns,
        context_columns=["method", "band"],
    )

    graph_stats_subject = graph_stats_subject.sort_values(
        ["q_fdr", "method", "band", "feature"]
    )

    save_table(
        graph_subject,
        OUT["tables"] / "extended_graph_features_subject_level.csv",
    )

    save_table(
        graph_stats_subject,
        OUT["tables"] / "extended_graph_stats_subject_level.csv",
    )

    print("Extended graph metrics subject-level:")
    display(
        graph_subject
        .groupby("group")["subject_id"]
        .nunique()
        .to_frame("n_subjects")
    )

    display(graph_stats_subject.head(30))

In [ ]:
# @title 11.9. Общая сводка статистики по блокам признаков

stats_tables = []

if "dfa_stats_subject" in globals():
    tmp = dfa_stats_subject.copy()
    tmp["feature_block"] = "DFA / LRTC"
    stats_tables.append(tmp)

if "burst_stats_subject" in globals():
    tmp = burst_stats_subject.copy()
    tmp["feature_block"] = "Burst"
    stats_tables.append(tmp)

if "connectivity_stats_subject" in globals():
    tmp = connectivity_stats_subject.copy()
    tmp["feature_block"] = "Connectivity"
    stats_tables.append(tmp)

if "event_stats_subject" in globals():
    tmp = event_stats_subject.copy()
    tmp["feature_block"] = "Events"
    stats_tables.append(tmp)

if "graph_stats_subject" in globals():
    tmp = graph_stats_subject.copy()
    tmp["feature_block"] = "Graph metrics"
    stats_tables.append(tmp)

if not stats_tables:
    raise ValueError("Не найдено ни одной статистической таблицы.")

all_temporal_stats_subject = pd.concat(
    stats_tables,
    ignore_index=True,
)

all_temporal_stats_subject = all_temporal_stats_subject.sort_values(
    ["q_fdr", "feature_block", "feature"]
)

stats_summary_by_block = (
    all_temporal_stats_subject
    .groupby("feature_block", as_index=False)
    .agg(
        n_tests=("feature", "count"),
        n_significant_fdr_0_05=("significant_fdr_0_05", "sum"),
        max_abs_hedges_g=("hedges_g", lambda x: np.nanmax(np.abs(x))),
        min_q_fdr=("q_fdr", "min"),
    )
    .sort_values("min_q_fdr")
)

save_table(
    all_temporal_stats_subject,
    OUT["tables"] / "all_temporal_connectivity_event_stats_subject_level.csv",
)

save_table(
    stats_summary_by_block,
    OUT["tables"] / "temporal_connectivity_event_stats_summary_by_block.csv",
)

print("Общая статистика сформирована.")
print("Всего тестов:", len(all_temporal_stats_subject))
print(
    "Значимых после FDR q < 0.05:",
    int(all_temporal_stats_subject["significant_fdr_0_05"].sum()),
)

display(stats_summary_by_block)
display(all_temporal_stats_subject.head(40))

In [ ]:
# @title 11.10. Топ временных, сетевых и событийных признаков по размеру эффекта

top_n = 40

top_temporal_effects = (
    all_temporal_stats_subject
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["hedges_g"])
    .copy()
)

top_temporal_effects["abs_hedges_g"] = top_temporal_effects[
    "hedges_g"
].abs()

top_temporal_effects = (
    top_temporal_effects
    .sort_values("abs_hedges_g", ascending=False)
    .head(top_n)
    .copy()
)

save_table(
    top_temporal_effects,
    OUT["tables"] / "top_temporal_connectivity_event_effects_subject_level.csv",
)

display(top_temporal_effects)

In [ ]:
# @title 11.11. Итоговый markdown-отчёт по временным, сетевым и событийным признакам

def p_to_stars(q_value: float) -> str:
    """Преобразует q-FDR в звёздочки значимости."""
    if pd.isna(q_value):
        return ""

    if q_value < 0.001:
        return "***"

    if q_value < 0.01:
        return "**"

    if q_value < 0.05:
        return "*"

    return ""


def format_float(value, digits: int = 3) -> str:
    """Форматирует числа для markdown-отчёта."""
    if pd.isna(value):
        return "NA"

    value = float(value)

    if value == 0:
        return "0"

    if abs(value) < 0.001 or abs(value) >= 10000:
        return f"{value:.3e}"

    return f"{value:.{digits}f}"


def format_median_iqr(median, iqr) -> str:
    """Форматирует медиану и IQR."""
    return f"{format_float(median)} [{format_float(iqr)}]"


def get_direction(delta_value: float) -> str:
    """Возвращает направление эффекта."""
    if pd.isna(delta_value):
        return "NA"

    if delta_value > 0:
        return "выше при ЧМТ"

    if delta_value < 0:
        return "ниже при ЧМТ"

    return "без разницы"


report_df = all_temporal_stats_subject.copy()

report_df["direction"] = report_df["delta_median_tbi_minus_control"].map(
    get_direction
)

report_df["significance"] = report_df["q_fdr"].map(p_to_stars)

report_df["tbi_median_iqr"] = report_df.apply(
    lambda row: format_median_iqr(row["tbi_median"], row["tbi_iqr"]),
    axis=1,
)

report_df["control_median_iqr"] = report_df.apply(
    lambda row: format_median_iqr(row["control_median"], row["control_iqr"]),
    axis=1,
)

report_df["abs_hedges_g"] = report_df["hedges_g"].abs()

top_report_df = (
    report_df
    .dropna(subset=["hedges_g"])
    .sort_values("abs_hedges_g", ascending=False)
    .head(40)
    .copy()
)

significant_report_df = report_df[
    report_df["significant_fdr_0_05"]
].copy()

report_dir = OUT["tables"] / "final_report_temporal_connectivity_events"
report_dir.mkdir(parents=True, exist_ok=True)

full_report_path = report_dir / "temporal_connectivity_event_stats_all.csv"
significant_report_path = (
    report_dir / "temporal_connectivity_event_stats_significant.csv"
)
top_report_path = report_dir / "temporal_connectivity_event_stats_top_effects.csv"
summary_path = report_dir / "temporal_connectivity_event_stats_summary_by_block.csv"

report_df.to_csv(full_report_path, index=False)
significant_report_df.to_csv(significant_report_path, index=False)
top_report_df.to_csv(top_report_path, index=False)
stats_summary_by_block.to_csv(summary_path, index=False)

markdown_lines = []

markdown_lines.append(
    "# Итоговый отчёт по временным, сетевым и событийным признакам"
)
markdown_lines.append("")
markdown_lines.append("## Уровень статистического анализа")
markdown_lines.append("")
markdown_lines.append(
    "Основная статистическая интерпретация выполнена на subject-level. "
    "Если у субъекта было несколько записей, признаки предварительно "
    "усреднялись внутри субъекта."
)
markdown_lines.append("")
markdown_lines.append(
    "Для межгруппового сравнения использовался Mann–Whitney U test. "
    "Размер эффекта оценивался с помощью Hedges' g и rank-biserial "
    "correlation. Коррекция на множественные сравнения выполнялась "
    "методом Benjamini–Hochberg FDR."
)
markdown_lines.append("")
markdown_lines.append("## Сводка по блокам признаков")
markdown_lines.append("")
markdown_lines.append(
    "| Блок признаков | Число тестов | Значимых q<0.05 | "
    "Макс. |Hedges' g| | Мин. q-FDR |"
)
markdown_lines.append("|---|---:|---:|---:|---:|")

for _, row in stats_summary_by_block.iterrows():
    markdown_lines.append(
        f"| {row['feature_block']} "
        f"| {int(row['n_tests'])} "
        f"| {int(row['n_significant_fdr_0_05'])} "
        f"| {format_float(row['max_abs_hedges_g'])} "
        f"| {format_float(row['min_q_fdr'])} |"
    )

markdown_lines.append("")
markdown_lines.append("## Топ-40 признаков по абсолютному размеру эффекта")
markdown_lines.append("")
markdown_lines.append(
    "| Блок | Контекст | Признак | ЧМТ, медиана [IQR] | "
    "Контроль, медиана [IQR] | Направление | Hedges' g | q-FDR |"
)
markdown_lines.append("|---|---|---|---:|---:|---|---:|---:|")

context_columns_possible = [
    "roi",
    "band",
    "method",
]

for _, row in top_report_df.iterrows():
    context_parts = []

    for col in context_columns_possible:
        if col in row.index and pd.notna(row[col]):
            context_parts.append(f"{col}={row[col]}")

    context_text = "; ".join(context_parts) if context_parts else "global"

    markdown_lines.append(
        f"| {row['feature_block']} "
        f"| {context_text} "
        f"| {row['feature']} "
        f"| {row['tbi_median_iqr']} "
        f"| {row['control_median_iqr']} "
        f"| {row['direction']} "
        f"| {format_float(row['hedges_g'])} "
        f"| {format_float(row['q_fdr'])}{row['significance']} |"
    )

markdown_lines.append("")
markdown_lines.append("## Примечание")
markdown_lines.append("")
markdown_lines.append(
    "Положительное значение Hedges' g означает, что признак выше в группе ЧМТ. "
    "Отрицательное значение означает, что признак ниже в группе ЧМТ "
    "по сравнению с контролем."
)

markdown_report_path = (
    report_dir / "temporal_connectivity_event_stats_report.md"
)

markdown_report_path.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8",
)

print("Итоговый статистический отчёт сформирован.")
print(f"- {full_report_path}")
print(f"- {significant_report_path}")
print(f"- {top_report_path}")
print(f"- {summary_path}")
print(f"- {markdown_report_path}")

display(stats_summary_by_block)
display(top_report_df)

# 12. Финальная проверка результатов

In [ ]:
# @title 12.1. Финальная проверка выходных файлов

required_outputs = {
    "dfa_features_long": OUT["tables"] / "dfa_features_long.csv",
    "burst_features_long": OUT["tables"] / "burst_features_long.csv",
    "connectivity_features_long": (
        OUT["tables"] / "connectivity_features_long.csv"
    ),
    "connectivity_matrices_long": (
        OUT["tables"] / "connectivity_matrices_long.csv"
    ),
    "extended_graph_features_long": (
        OUT["tables"] / "extended_graph_features_long.csv"
    ),
    "event_features": OUT["tables"] / "event_features.csv",
    "event_manifest_long": OUT["tables"] / "event_manifest_long.csv",
    "event_features_extended": (
        OUT["tables"] / "event_features_extended.csv"
    ),
    "temporal_connectivity_event_features_record_level": (
        OUT["tables"]
        / "temporal_connectivity_event_features_record_level.csv"
    ),
    "feature_map": (
        OUT["tables"] / "temporal_connectivity_event_feature_map.csv"
    ),
    "missingness": (
        OUT["tables"]
        / "temporal_connectivity_event_feature_missingness.csv"
    ),
    "subject_level_features": (
        OUT["tables"]
        / "temporal_connectivity_event_features_subject_level.csv"
    ),
    "subject_level_stats": (
        OUT["tables"]
        / "all_temporal_connectivity_event_stats_subject_level.csv"
    ),
}

print("Проверка ключевых выходных файлов")
print("-" * 70)

for name, path in required_outputs.items():
    status = "OK" if path.exists() else "НЕ НАЙДЕН"
    print(f"{name}: {status}")
    print(f"  {path}")

print("\nВременной, сетевой и событийный анализ завершён.")
